# Xa-BB 9-gene — Pipeline Colab (homology anchor + %N + SVM + mCNN)

Dự đoán tính kháng bạc lá (Xoo) từ 327 FASTA, neo theo **9 gene kháng đã có trình tự** (align homology → tránh trượt tọa độ), lọc **%N**, embedding **DNABERT**, so sánh **SVM / classical / mCNN**, đánh giá có kiểm soát cấu trúc quần thể (**GroupKFold**).

| Phần | Việc | Ra |
|---|---|---|
| 0 | Cài đặt + mount Drive + config | |
| 1 | Kiểu hình 4 chủng + nhãn nhị phân + subgroup | |
| 2 | Khớp 327 FASTA ↔ kiểu hình | |
| 3 | Dựng FASTA query 9 gene từ Excel | |
| 4 | Kiểm tra trượt tọa độ (câu hỏi thầy) | Bảng trả lời |
| 5 | **Neo homology 9 gene** vào từng mẫu (mappy) → trích locus | drift ~0 |
| 6 | Lọc **%N** | |
| 7 | Embedding DNABERT theo locus | embeddings.npz |
| 8 | Ghép ma trận đặc trưng | X_emb (n × 9·768) |
| 9 | Module đánh giá chuẩn (randomCV + GroupKFold, đủ chỉ số) | |
| 10 | Baseline: LogReg / Ridge / RF / XGB | |
| 11 | **SVM** (linear + RBF) | |
| 12 | **mCNN** trên DNA one-hot (50 epoch, N lần lặp) | |
| 13 | Bảng tổng hợp | metrics_all.csv |

Mỗi phần kết bằng `CHECKPOINT`. Gặp FAIL thì dừng, đọc ghi chú, sửa rồi chạy tiếp.

## 0. Cài đặt + mount Drive

In [42]:
# Chạy 1 lần mỗi phiên Colab. Runtime -> Change runtime type -> GPU (T4 là đủ).
!pip -q install mappy pyfaidx openpyxl xgboost 2>/dev/null
# transformers cho DNABERT/Plant-DNABERT (bản khớp einops)
!pip -q install "transformers==4.44.2" einops accelerate 2>/dev/null
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

The system cannot find the path specified.


CUDA: True | NVIDIA GeForce RTX 5060 Ti


The system cannot find the path specified.


In [43]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

### 0b. CONFIG — SỬA CÁC ĐƯỜNG DẪN NÀY CHO KHỚP DRIVE CỦA BẠN

In [ ]:
import os
# ====== SỬA CHO KHỚP DRIVE ======
ROOT       = '/content/drive/MyDrive/Project/Bioinformatics - Thesis'
FASTA_DIR  = '/content/drive/MyDrive/consensus_results'
PHENO_XLS  = f'{ROOT}/Bacterial Leaf Data/12284_2021_462_MOESM2_ESM.xlsx'
GENE9_XLS  = f'{ROOT}/Bacterial Leaf Data/9_gene_BB_thong_tin_khoa_hoc_va_trinh_tu.xlsx'
# ================================
OUT     = f'{ROOT}/Export'; os.makedirs(OUT, exist_ok=True)
TMP     = '/content/work'; os.makedirs(TMP, exist_ok=True)
OUT_SEQ = f'{OUT}/regions_seq'; os.makedirs(OUT_SEQ, exist_ok=True)

SEED        = 42
STRAINS     = ['C4', 'C5', 'V', 'P9a']
BIN_STRAINS = ['C4', 'C5', 'V', 'P9a']   # GIỮ cả V; mất cân bằng xử lý bằng class_weight (dưới)
#MAX_PCTN    = 15.0
MAX_PCTN_GENE   = 10.0    # QC ở mức gene
MAX_PCTN_SAMPLE = 20.0    # QC ở mức accession
FLANK       = 2000

# --- mất cân bằng lớp (cho chủng V) ---
USE_CLASS_WEIGHT = True   # bật class_weight='balanced' (model cổ điển) + pos_weight (neural)
MIN_MINORITY     = 8      # cảnh báo nếu 1 chủng có < ngưỡng này mẫu lớp thiểu số

# --- embedding ---
EMB_MODEL   = 'plant-dnabert'
POOL        = 'mean'
# --- mCNN ---
MCNN_EPOCHS   = 50
MCNN_LOCUS_MAXLEN = 1500
MCNN_BATCH    = 16
GENO_DIR_NOTE = "GENO_DIR (.bed/.bim/.fam) khai báo ở mục 12.3"

for p in [PHENO_XLS, GENE9_XLS]:
    print(('OK    ' if os.path.exists(p) else 'THIẾU ') + os.path.basename(p))
print(('OK    ' if os.path.isdir(FASTA_DIR) else 'THIẾU ') + 'FASTA_DIR')
print('Output ->', OUT)

OK    12284_2021_462_MOESM2_ESM.xlsx
OK    9_gene_BB_thong_tin_khoa_hoc_va_trinh_tu.xlsx
OK    FASTA_DIR
Output -> /content/drive/MyDrive/Project/Bioinformatics - Thesis/Export


## 0c. Đường dẫn từ máy local

In [ ]:
import os
# ====== ĐƯỜNG DẪN WINDOWS (máy thuê) ======
ROOT       = r'C:\GenBBPipeline'
OUT        = rf'{ROOT}\Export'                              # file đã chạy trên Colab
DATA_DIR   = rf'{ROOT}\Bacterial Leaf Data'                 # genotype + excel
PHENO_XLS  = rf'{DATA_DIR}\12284_2021_462_MOESM2_ESM.xlsx'
GENE9_XLS  = rf'{DATA_DIR}\9_gene_BB_thong_tin_khoa_hoc_va_trinh_tu.xlsx'
OUT_SEQ    = rf'{OUT}\regions_seq'
TMP        = rf'{ROOT}\work'; os.makedirs(TMP, exist_ok=True)
# KHÔNG cần FASTA_DIR trên máy thuê (không align lại)
# ==========================================

SEED        = 42
STRAINS     = ['C4', 'C5', 'V', 'P9a']
BIN_STRAINS = ['C4', 'C5', 'P9a']   # V loại khỏi phân loại (mất cân bằng nặng)
MAX_PCTN_GENE   = 10.0
MAX_PCTN_SAMPLE = 20.0
FLANK       = 2000
USE_CLASS_WEIGHT = True
EMB_MODEL   = 'plant-dnabert'
POOL        = 'mean'

# kiểm file có mặt
for p in [OUT, OUT_SEQ, DATA_DIR]:
    print(('OK    ' if os.path.exists(p) else 'THIẾU ') + p)
for p in [PHENO_XLS, GENE9_XLS]:
    print(('OK    ' if os.path.exists(p) else 'THIẾU ') + os.path.basename(p))

OK    C:\GenBBPipeline\Export
OK    C:\GenBBPipeline\Export\regions_seq
OK    C:\GenBBPipeline\Bacterial Leaf Data
OK    12284_2021_462_MOESM2_ESM.xlsx
OK    9_gene_BB_thong_tin_khoa_hoc_va_trinh_tu.xlsx


## 0d. Nạp genotype để train SNP

In [ ]:
from bed_reader import open_bed
GENO_PREFIX = r'C:\GenBBPipeline\Bacterial Leaf Data\genotype\base_filtered_v0.7'
bed = open_bed(GENO_PREFIX + '.bed')
print('Số mẫu (rows):', bed.iid_count)
print('Số SNP (cols):', bed.sid_count)
print('Vài ID mẫu đầu:', bed.iid[:5])

Số mẫu (rows): 3024
Số SNP (cols): 4817964
Vài ID mẫu đầu: ['B001' 'B002' 'B003' 'B004' 'B005']


## 1. Kiểu hình 4 chủng + nhãn nhị phân + subgroup
Đọc file bổ trợ Lu 2021. Nhãn nhị phân: R/MR = 1 (kháng), MS/S = 0 (nhiễm). `subgroup` = nhóm quần thể, dùng làm **group** cho GroupKFold.

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore'); pd.set_option('display.width',210); pd.set_option('display.max_columns',None)

d = pd.read_excel(PHENO_XLS, header=None, skiprows=3)
d.columns = ['id','name','origin','region','subgroup',
             'C4_lvl','C4_LL','C4_SD','C5_lvl','C5_LL','C5_SD',
             'V_lvl','V_LL','V_SD','P9a_lvl','P9a_LL','P9a_SD']
d = d.dropna(subset=['id','subgroup']); d['id'] = d['id'].astype(str).str.strip()

# R/MR = 1 (kháng), MS/S = 0 (nhiễm); chuẩn hoá hoa/thường, thêm HR/HS nếu có
LVL2BIN = {'R':1,'MR':1,'MS':0,'S':0,'HR':1,'HS':0}
def to_bin(x): return LVL2BIN.get(str(x).strip().upper(), np.nan)
for s in STRAINS:
    d[s+'_bin'] = d[s+'_lvl'].map(to_bin)
    d[s+'_LL']  = pd.to_numeric(d[s+'_LL'], errors='coerce')

# giữ mẫu đủ LL và đủ nhãn nhị phân cho TẤT CẢ chủng (gồm V)
pheno = d.dropna(subset=[s+'_LL' for s in STRAINS]+[s+'_bin' for s in BIN_STRAINS]).reset_index(drop=True)
for s in BIN_STRAINS: pheno[s+'_bin'] = pheno[s+'_bin'].astype(int)

# --- báo cáo cân bằng lớp ---
rowsb=[]
for s in STRAINS:
    npos=int(pheno[s+'_bin'].sum()); nneg=len(pheno)-npos; minority=min(npos,nneg)
    rowsb.append({'strain':s,'khang(1)':npos,'nhiem(0)':nneg,'%khang':round(100*npos/len(pheno),1),
                  'thiểu số':minority,'~thiểu số/fold(5)':round(minority/5,1)})
bal=pd.DataFrame(rowsb)
print(f'n = {len(pheno)} | subgroup = {pheno.subgroup.nunique()}\n')
print(bal.to_string(index=False))
for r in rowsb:
    if r['thiểu số'] < MIN_MINORITY:
        print(f"\n⚠ {r['strain']}: chỉ {r['thiểu số']} mẫu lớp thiểu số -> StratifiedKFold vẫn chạy, "
              f"nhưng GroupKFold có thể có fold thiếu lớp (fold đó bị bỏ). Đọc kỹ số fold khi đánh giá V, "
              f"và nhìn MCC/AUC/F1 thay vì Accuracy.")
pheno.to_csv(f'{OUT}/pheno.csv', index=False)
print('\nCHECKPOINT 1:', 'PASS' if len(pheno)>=300 else 'FAIL')

n = 340 | subgroup = 12

strain  khang(1)  nhiem(0)  %khang  thiểu số  ~thiểu số/fold(5)
    C4       171       169    50.3       169               33.8
    C5        66       274    19.4        66               13.2
     V        27       313     7.9        27                5.4
   P9a       173       167    50.9       167               33.4

CHECKPOINT 1: PASS


## 2. Khớp FASTA ↔ kiểu hình

In [ ]:
import glob
pats=['*.fa','*.fasta','*.fna','*.fa.gz','*.fasta.gz','*.fna.gz']
files=sorted(set(sum([glob.glob(f'{FASTA_DIR}/{p}') for p in pats],[])))
def stem(p):
    b=os.path.basename(p)
    for e in ['.fasta.gz','.fa.gz','.fna.gz','.fasta','.fa','.fna']:
        if b.endswith(e): return b[:-len(e)]
    return os.path.splitext(b)[0]
fmap={stem(f):f for f in files}
pheno['fasta']=pheno['id'].map(fmap)
matched=pheno.dropna(subset=['fasta']).reset_index(drop=True)
print(f'FASTA={len(files)} | có pheno={len(pheno)} | khớp={len(matched)}')
if len(matched)<len(pheno):
    print('Thiếu:', pheno.loc[pheno.fasta.isna(),'id'].tolist()[:15])
matched.to_csv(f'{OUT}/matched.csv', index=False)
df_snp = matched.copy()   # tên dùng lại xuyên suốt (giữ thứ tự mẫu)
print('CHECKPOINT 2:', 'PASS' if len(matched)>=250 else 'FAIL')

FASTA=346 | có pheno=340 | khớp=327
Thiếu: ['CX182', 'CX219', 'IRIS_313-9182', 'IRIS_313-11257', 'IRIS_313-11817', 'IRIS_313-8151', 'IRIS_313-11689', 'IRIS_313-8173', 'IRIS_313-12060', 'IRIS_313-11047', 'IRIS_313-11048', 'IRIS_313-11742', 'IRIS_313-11258']
CHECKPOINT 2: PASS


## 3. Dựng FASTA query 9 gene từ Excel
Đọc header (cột *FASTA header*) + trình tự (cột *Trình tự DNA 5′→3′*). Mỗi query kèm `preset` cho mappy theo loại trình tự (mRNA/CDS → `splice`; genomic/anchor → `asm20`).

In [ ]:
import re
# =====================================================================
# CACH HOAT DONG: cell nay doc SEQUENCE (khong phai toa do) tu Excel,
# dung lam QUERY de align vao tung genome mau (muc 5) -> tim vi tri -> trich.
#
# PHAN LOAI LOCUS:
#   NHOM A (co gene-model tren IRGSP, trich nhu binh thuong):
#       xa5, xa13, xa25, Xa1, Xa3/Xa26, Xa21, Xa27  (+ Xa4, xa41, Xa47 neu them)
#   NHOM B (VANG MAT tren Nipponbare -> KHONG trich theo cach nay,
#           phai dung presence/absence o muc 3b.2):
#       Xa7, Xa10, Xa23, Xa48
#
# CANH BAO mRNA: cac query la mRNA (xa5, xa13, Xa1, Xa7) chi chua exon
#   -> khi align co the THIEU promoter/intron. Voi xa5/xa13 (khang qua
#   promoter), day la han che -> ket qua embedding can dien giai than trong.
#   Neu co accession GENOMIC thay the thi nen dung.
# =====================================================================

# Danh sach locus Nhom B (vang mat Nipponbase) — xu ly rieng, KHONG trich sequence
GROUP_B_LOCI = {'Xa7', 'Xa10', 'Xa23', 'Xa48'}
# Query la mRNA (thieu promoter/intron) — canh bao khi dung
MRNA_QUERIES = {'Xa1', 'xa5', 'Xa7', 'xa13_SWEET11'}

wb = pd.ExcelFile(GENE9_XLS)
g9 = pd.read_excel(GENE9_XLS, sheet_name=0, header=4)   # header thật ở dòng 5
g9.columns = [str(c).strip() for c in g9.columns]
COL_GENE='Gene / locus'; COL_TYPE='Loại sequence được cung cấp'
COL_HDR='FASTA header';   COL_SEQ='Trình tự DNA 5′→3′'
g9 = g9.dropna(subset=[COL_SEQ]).reset_index(drop=True)

def clean_seq(x):
    body=''.join(l for l in str(x).split('\n') if not l.startswith('>'))
    return re.sub(r'\s','',body).upper()
def region_id(gene):
    return re.sub(r'[^A-Za-z0-9]+','_', str(gene)).strip('_')   # 'xa13/SWEET11' -> 'xa13_SWEET11'
def preset_of(t):
    t=str(t).lower()
    is_spliced = ('mrna' in t or 'cdna' in t) or ('cds' in t and 'genomic' not in t)
    return 'splice' if is_spliced else 'asm20'
def group_of(rid):
    return 'B' if rid in GROUP_B_LOCI else 'A'

QUERIES=[]   # list of dict(rid, seq, preset, gene, group)
qfasta=f'{OUT}/query_9gene.fasta'
with open(qfasta,'w') as fo:
    for _, r in g9.iterrows():
        gene=r[COL_GENE]; seq=clean_seq(r[COL_SEQ]); typ=r[COL_TYPE]
        rid=region_id(gene)
        grp=group_of(rid)
        QUERIES.append(dict(rid=rid, seq=seq, preset=preset_of(typ), gene=str(gene), group=grp))
        fo.write(f'>{rid}\n{seq}\n')

REG_IDS=[q['rid'] for q in QUERIES]
# Nhom A dung de trich sequence + embedding; Nhom B chuyen sang presence/absence
REG_IDS_A=[q['rid'] for q in QUERIES if q['group']=='A']
REG_IDS_B=[q['rid'] for q in QUERIES if q['group']=='B']

print(f'{len(QUERIES)} query -> {qfasta}')
for q in QUERIES:
    flag=''
    if q['rid'] in GROUP_B_LOCI: flag=' [NHOM B: presence/absence]'
    elif q['rid'] in MRNA_QUERIES: flag=' [mRNA: thieu promoter/intron]'
    print(f"  {q['rid']:16s} len={len(q['seq']):6d} preset={q['preset']:6s} nhom={q['group']}{flag}")
print(f'\nNhom A (trich+embedding): {REG_IDS_A}')
print(f'Nhom B (presence/absence): {REG_IDS_B}')
print('CHECKPOINT 3:', 'PASS' if len(QUERIES)>=8 else 'FAIL')


9 query -> /content/drive/MyDrive/Project/Bioinformatics - Thesis/Export/query_9gene.fasta
  Xa1              len=  5910 preset=splice nhom=A [mRNA: thieu promoter/intron]
  Xa21             len=  3921 preset=asm20  nhom=A
  Xa3_Xa26         len=  6001 preset=asm20  nhom=A
  xa5              len=  8727 preset=asm20  nhom=A [mRNA: thieu promoter/intron]
  Xa10             len=  1032 preset=asm20  nhom=B [NHOM B: presence/absence]
  xa13_SWEET11     len=  4654 preset=asm20  nhom=A [mRNA: thieu promoter/intron]
  Xa23             len=  1720 preset=asm20  nhom=B [NHOM B: presence/absence]
  xa25_SWEET13     len=  3208 preset=asm20  nhom=A
  Xa27             len=  2361 preset=asm20  nhom=A

Nhom A (trich+embedding): ['Xa1', 'Xa21', 'Xa3_Xa26', 'xa5', 'xa13_SWEET11', 'xa25_SWEET13', 'Xa27']
Nhom B (presence/absence): ['Xa10', 'Xa23']
CHECKPOINT 3: PASS


#3b. Xác nhận các accession đang có

In [ ]:
!pip install -q biopython

In [ ]:
# === VERIFY ACCESSION trên NCBI (chạy trên Colab) ===
# Kiem tra tung accession co ton tai khong, loai sequence gi, dai bao nhieu.
import Bio.Entrez, Bio.SeqIO
from Bio import Entrez, SeqIO
Entrez.email = 'socthanhnhi@gmail.com'   # <<< SUA email that (NCBI bat buoc)

# Danh sach tu ke hoach teammate (muc 5). Xa4/xa41/Xa47 CHUA co -> can audit.
ACCESSIONS = {
    'Xa1'          : 'AB002266',
    'Xa3_Xa26'     : 'DQ426645.1',
    'xa5'          : 'NM_001402467.1',
    'Xa7'          : 'MW467883',
    'Xa10'         : 'JX025645.1',
    'xa13_SWEET11' : 'NM_001403670.1',
    'Xa21'         : 'U37133',
    'Xa23'         : 'KP123634.1',
    'Xa27'         : 'AY986492.1',
    'xa25_SWEET13' : 'KC915031.1',
    'Xa48'         : 'OR712925',
    # --- CAN AUDIT (chua co accession xac nhan) ---
    'Xa4'          : None,
    'xa41_SWEET14' : None,
    'Xa47'         : None,
}

import time
rows=[]
for gene, acc in ACCESSIONS.items():
    if acc is None:
        rows.append(dict(gene=gene, accession='(chua co)', status='CAN AUDIT',
                         length='-', moltype='-')); continue
    try:
        h=Entrez.efetch(db='nucleotide', id=acc, rettype='gb', retmode='text')
        rec=SeqIO.read(h, 'genbank'); h.close()
        # phan loai loai sequence
        mol=rec.annotations.get('molecule_type','?')
        desc=rec.description[:45]
        is_mrna = 'mRNA' in mol or 'RNA' in mol
        rows.append(dict(gene=gene, accession=acc, status='OK',
                         length=len(rec.seq), moltype=mol,
                         note='mRNA/cDNA (thieu intron/promoter)' if is_mrna else 'genomic',
                         desc=desc))
        print(f'  OK  {gene:14s} {acc:16s} len={len(rec.seq):7d} {mol}')
        time.sleep(0.4)   # ton trong rate limit NCBI
    except Exception as e:
        rows.append(dict(gene=gene, accession=acc, status=f'LOI: {e}',
                         length='-', moltype='-'))
        print(f'  LOI {gene:14s} {acc}: {e}')

import pandas as pd
audit=pd.DataFrame(rows)
audit.to_csv(f'{OUT}/accession_audit.csv', index=False)
print('\n=== BANG AUDIT ===')
print(audit.to_string(index=False))
print('\nLUU Y: dong "mRNA/cDNA" KHONG dung cho embedding promoter/intron.')
print('       Can tim accession genomic DNA thay the cho cac dong do.')

  OK  Xa1            AB002266         len=   5910 mRNA
  OK  Xa3_Xa26       DQ426645.1       len=   6001 DNA
  OK  xa5            NM_001402467.1   len=    994 mRNA
  OK  Xa7            MW467883         len=    726 mRNA
  OK  Xa10           JX025645.1       len=   1032 DNA
  OK  xa13_SWEET11   NM_001403670.1   len=   1571 mRNA
  OK  Xa21           U37133           len=   3921 DNA
  OK  Xa23           KP123634.1       len=   1720 DNA
  OK  Xa27           AY986492.1       len=   2361 DNA
  OK  xa25_SWEET13   KC915031.1       len=   3208 DNA
  OK  Xa48           OR712925         len=   7860 DNA

=== BANG AUDIT ===
        gene      accession    status length moltype                              note                                          desc
         Xa1       AB002266        OK   5910    mRNA mRNA/cDNA (thieu intron/promoter)       Oryza sativa mRNA for XA1, complete cds
    Xa3_Xa26     DQ426645.1        OK   6001     DNA                           genomic Oryza sativa (indica cultivar

## 3c. MỞ RỘNG 14 LOCUS (tùy chọn — cho luận văn)

9 gene hiện tại → mở rộng thành **14 locus** theo kế hoạch. Chia 2 nhóm:

- **Nhóm A** (có tọa độ IRGSP, trích như 9 gene): thêm Xa4, xa41/SWEET14, Xa47.
  Cần accession GenBank → tải sequence → thêm vào QUERIES.
- **Nhóm B** (vắng mặt Nipponbase, cần presence/absence): Xa7, Xa10, Xa23, Xa48.
  Align donor accession vào TỪNG mẫu, xác định có/không có gen.

> Chạy mục này SAU mục 3 (đã có QUERIES 9 gene) và TRƯỚC mục 5 (neo homology).
> Nếu chỉ làm 9 gene, BỎ QUA mục 3b này.

In [ ]:
# === 3c.1 — Nhóm A: thêm locus có accession GenBank (Xa4, xa41, Xa47) ===
# Tải sequence từ NCBI Entrez, thêm vào QUERIES (dùng chung pipeline 9 gene).
from Bio import Entrez, SeqIO
Entrez.email = 'socthanhnhi@gmail.com'   # <<< SỬA email (NCBI yêu cầu)

# accession đại diện cho từng locus bổ sung (từ kế hoạch/tài liệu)
GROUP_A_EXTRA = {
    'Xa4'          : 'MW034470',    # WAK, Hu et al. 2017 (kiểm lại accession cho đúng)
    'xa41_SWEET14' : 'HG994060',    # OsSWEET14 promoter region (kiểm lại)
    'Xa47'         : None,          # <<< chưa có accession chuẩn -> cần audit/tạo query
}

def fetch_ncbi(acc):
    h=Entrez.efetch(db='nucleotide', id=acc, rettype='fasta', retmode='text')
    rec=SeqIO.read(h, 'fasta'); h.close()
    return str(rec.seq).upper()

added=0
for gene, acc in GROUP_A_EXTRA.items():
    if acc is None:
        print(f'  BỎ QUA {gene}: chưa có accession (cần audit)'); continue
    rid=region_id(gene)
    if rid in REG_IDS:
        print(f'  {gene}: đã có, bỏ qua'); continue
    try:
        seq=fetch_ncbi(acc)
        QUERIES.append(dict(rid=rid, seq=seq, preset='asm20', gene=gene))
        with open(f'{OUT}/query_9gene.fasta','a') as fo: fo.write(f'>{rid}\n{seq}\n')
        added+=1; print(f'  + {gene} ({acc}) len={len(seq)}')
    except Exception as e:
        print(f'  LỖI {gene} ({acc}): {e}')

REG_IDS=[q['rid'] for q in QUERIES]
print(f'\nNhóm A: thêm {added} locus. Tổng QUERIES = {len(QUERIES)}')

  + Xa4 (MW034470) len=702
  + xa41_SWEET14 (HG994060) len=1030
  BỎ QUA Xa47: chưa có accession (cần audit)

Nhóm A: thêm 2 locus. Tổng QUERIES = 11


### 3c.2 — Nhóm B: presence/absence cho gen vắng mặt Nipponbare

Xa7, Xa10, Xa23, Xa48 **không có trên Nipponbare** → không neo theo tọa độ IRGSP được.
Thay vào đó: align donor accession vào **từng** consensus genome, xác định gen **có mặt hay không**.
Kết quả là đặc trưng nhị phân (0/1) cho mỗi mẫu × mỗi locus B — ghép vào ma trận đặc trưng.

In [ ]:
import os
p = f'{OUT}/pa_progress.csv'
if os.path.exists(p): os.remove(p); print('Da xoa', p)
else: print('Khong co file (OK)')

Khong co file (OK)


In [ ]:
# === 3c.2 — presence/absence (Xa23+Xa48, dọn ổ đĩa) ===
from Bio import Entrez, SeqIO
import mappy as mp, os, time, shutil, gzip
import pandas as pd
Entrez.email = 'socthanhnhi@gmail.com'

def to_local(drive_path):
    base=os.path.basename(drive_path)
    local = f'{TMP}/{base[:-3]}' if base.endswith('.gz') else f'{TMP}/{base}'
    if not os.path.exists(local):
        if base.endswith('.gz'):
            with gzip.open(drive_path,'rb') as fi, open(local,'wb') as fo:
                shutil.copyfileobj(fi, fo)
        else: shutil.copy(drive_path, local)
    return local

GROUP_B = {'Xa23':'KP123634.1', 'Xa48':'OR712925'}   # đã bỏ Xa7, Xa10
PA_MIN_COV, PA_MIN_IDT = 0.60, 0.80

donorB={}
for gene, acc in GROUP_B.items():
    h=Entrez.efetch(db='nucleotide', id=acc, rettype='fasta', retmode='text')
    donorB[gene]=str(SeqIO.read(h,'fasta').seq).upper(); h.close()
    print(f'  {gene} ({acc}) len={len(donorB[gene])}')

CKPT_PA=f'{OUT}/pa_progress.csv'
if os.path.exists(CKPT_PA):
    prev=pd.read_csv(CKPT_PA); pa_rows=prev.to_dict('records')
    done_ids=set(prev.id.unique()); print(f'Resume: đã có {len(done_ids)} mẫu')
else:
    pa_rows=[]; done_ids=set()

t0=time.time()
for n, r in enumerate(matched.itertuples(), 1):
    if r.id in done_ids: continue
    loc=None
    try:
        loc=to_local(r.fasta); a=mp.Aligner(loc, preset='asm20')
    except Exception as e:
        print(f'  LỖI {r.id}: {e}'); a=None
    for gene, dseq in donorB.items():
        best=(0.0,0.0)
        if a:
            for hit in a.map(dseq):
                cov=(hit.q_en-hit.q_st)/max(1,len(dseq)); idt=hit.mlen/max(1,hit.blen)
                if cov*idt>best[0]*best[1]: best=(cov,idt)
        pa_rows.append(dict(id=r.id, locus=gene,
            present=int(best[0]>=PA_MIN_COV and best[1]>=PA_MIN_IDT),
            cov=round(best[0],3), idt=round(best[1],3)))
    del a                                    # giải phóng index
    if loc and os.path.exists(loc): os.remove(loc)   # <<< XÓA file, dọn ổ
    if n % 25 == 0:
        pd.DataFrame(pa_rows).to_csv(CKPT_PA, index=False)
        print(f'  {n}/{len(matched)} mẫu ({time.time()-t0:.0f}s)', flush=True)

pa_df=pd.DataFrame(pa_rows)
pa_df.to_csv(f'{OUT}/presence_absence_groupB.csv', index=False)
PA_MAT=pa_df.pivot(index='id', columns='locus', values='present').fillna(0).astype(int)
print('\nPresence/absence:', PA_MAT.shape)
print(PA_MAT.sum().to_string())
print(f'Thời gian: {time.time()-t0:.0f}s')

  Xa23 (KP123634.1) len=1720
  Xa48 (OR712925) len=7860
Resume: đã có 125 mẫu
  150/327 mẫu (449s)
  175/327 mẫu (908s)
  200/327 mẫu (1384s)
  225/327 mẫu (1846s)
  250/327 mẫu (2364s)
  275/327 mẫu (2856s)
  300/327 mẫu (3330s)
  325/327 mẫu (3808s)

Presence/absence: (327, 4)
locus
Xa10      0
Xa23    259
Xa48     50
Xa7       0
Thời gian: 3849s


In [ ]:
import glob, os
n=0
for f in glob.glob(f'{TMP}/*.fasta')+glob.glob(f'{TMP}/*.fa')+glob.glob(f'{TMP}/*.fasta.fai')+glob.glob(f'{TMP}/*.mmi'):
    os.remove(f); n+=1
print(f'Da xoa {n} file')
!df -h /content | tail -1

Da xoa 139 file
overlay         113G   51G   63G  45% /


In [ ]:
def aligner_for(loc, dseq_len):
    # sequence ngan (mRNA) can preset nhay hon
    preset = 'asm20' if dseq_len >= 1500 else 'splice'
    return mp.Aligner(loc, preset=preset)

> **Ghép PA_MAT vào đặc trưng:** ở mục 8 (ghép ma trận), nối `PA_MAT` (4 cột nhị phân)
> vào `X_emb` để thành đặc trưng đầy đủ 14 locus (10 embedding + 4 presence/absence).
> Ví dụ: `X_full = np.hstack([X_emb, PA_MAT.loc[ids_keep].values])`.

#3d. Query genomic có promoter từ IRGSP-1.0 (chuẩn sinh học)

Các gene kháng **qua biến dị promoter** (xa5, xa13/SWEET11, xa25/SWEET13, xa41/SWEET14)
nếu dùng query mRNA sẽ **mất vùng promoter** — nơi chứa tín hiệu kháng thật.

Cell này thay query của các gene đó bằng **vùng gene + promoter** trích từ IRGSP-1.0
theo tọa độ Excel (nới upstream theo chiều gene). Khi align vào genome mẫu,
nó trích đúng vùng promoter **của từng mẫu** — giữ được biến dị kháng/nhiễm.

> Cần: `IRGSP_FASTA` trỏ tới reference trên Drive; Excel có cột tọa độ + strand.

In [ ]:
# === 3d — Thay query mRNA bang query genomic co promoter (tu IRGSP-1.0) ===
from pyfaidx import Fasta

IRGSP_FASTA = f'{ROOT}/reference/IRGSP-1.0_genome.fasta'   # <<< SUA cho khop ten file tren Drive
PROMO_UP  = 1500   # bp promoter phia upstream
GENE_PAD  = 200    # noi nhe gene body

# gene khang-qua-promoter -> can noi upstream
PROMOTER_GENES = {'xa5', 'xa13_SWEET11', 'xa25_SWEET13', 'xa41_SWEET14'}

# doc lai toa do tu Excel (cot IRGSP Start/End/Chromosome/Strand)
g9c = pd.read_excel(GENE9_XLS, sheet_name=0, header=4)
g9c.columns=[str(c).strip() for c in g9c.columns]
COL_CHR='Chromosome'; COL_ST='IRGSP Start'; COL_EN='IRGSP End'; COL_STRAND='Strand'

# map ten chromosome IRGSP (thu vai kieu dat ten pho bien)
irgsp = Fasta(IRGSP_FASTA)
irgsp_keys = list(irgsp.keys())
def find_chrom(ch):
    ch=str(ch).replace('chr','').replace('Chr','').strip()
    for cand in [f'chr{ch:0>2}', f'chr{ch}', f'Chr{ch}', ch, f'Chr{ch:0>2}',
                 f'chr0{ch}' if len(ch)==1 else f'chr{ch}']:
        if cand in irgsp_keys: return cand
    # thu so khop long
    for k in irgsp_keys:
        if k.lower().endswith(str(ch).lower()): return k
    return None

def revcomp(s):
    return s.translate(str.maketrans('ACGTNacgtn','TGCANtgcan'))[::-1]

replaced=0
for q in QUERIES:
    if q['rid'] not in PROMOTER_GENES: continue
    row=g9c[g9c['Gene / locus'].apply(lambda x: region_id(x))==q['rid']]
    if row.empty: print(f'  {q["rid"]}: khong thay toa do, giu query cu'); continue
    r=row.iloc[0]
    ch=find_chrom(r[COL_CHR])
    if ch is None or pd.isna(r[COL_ST]) or pd.isna(r[COL_EN]):
        print(f'  {q["rid"]}: thieu toa do/chromosome, giu query cu'); continue
    st=int(r[COL_ST]); en=int(r[COL_EN]); strand=str(r.get(COL_STRAND,'+')).strip()
    L=len(irgsp[ch])
    if strand=='-':
        s2=max(0, st-GENE_PAD); e2=min(L, en+PROMO_UP)   # promoter o phia end
        seq=revcomp(str(irgsp[ch][s2:e2]))
    else:
        s2=max(0, st-PROMO_UP); e2=min(L, en+GENE_PAD)   # promoter o phia start
        seq=str(irgsp[ch][s2:e2])
    old_len=len(q['seq'])
    q['seq']=seq.upper(); q['preset']='asm20'   # genomic -> asm20 (khong con splice)
    q['note']='genomic+promoter tu IRGSP'
    replaced+=1
    print(f'  {q["rid"]:16s} mRNA {old_len}bp -> genomic+promoter {len(seq)}bp ({ch}:{s2}-{e2} {strand})')

# ghi lai query FASTA da cap nhat
with open(qfasta,'w') as fo:
    for q in QUERIES: fo.write(f'>{q["rid"]}\n{q["seq"]}\n')
print(f'\nDa thay {replaced} query sang genomic+promoter. Query FASTA cap nhat.')
print('Cac gene con lai giu nguyen (da la genomic hoac thuoc Nhom B).')

  xa5              mRNA 8727bp -> genomic+promoter 7926bp (5:435543-443469 +)
  xa13_SWEET11     mRNA 4654bp -> genomic+promoter 4553bp (8:26725754-26730307 -)
  xa25_SWEET13     mRNA 3208bp -> genomic+promoter 4899bp (12:17301927-17306826 -)
  xa41_SWEET14: khong thay toa do, giu query cu

Da thay 3 query sang genomic+promoter. Query FASTA cap nhat.
Cac gene con lai giu nguyen (da la genomic hoac thuoc Nhom B).


## 4. Kiểm tra trượt tọa độ (câu hỏi của thầy)
So chiều dài 12 nhiễm sắc thể giữa các mẫu. Nếu chiều dài lệch nhau → **không** được cắt theo tọa độ cố định. Đây là lý do phần 5 dùng **neo homology** (align trình tự), miễn nhiễm với trượt tọa độ.

In [ ]:
from pyfaidx import Fasta
import gzip, shutil
CHRLEN = {f'chr{i}': v for i,v in enumerate(
    [43270923,35937250,36413819,35502694,29958434,31248787,
     29697621,28443022,23012720,23207287,29021106,27531856],1)}

def to_local(src):
    b=os.path.basename(src)
    dst=f'{TMP}/'+(b[:-3] if b.endswith('.gz') else b)
    if not os.path.exists(dst):
        if b.endswith('.gz'):
            with gzip.open(src,'rb') as fi, open(dst,'wb') as fo: shutil.copyfileobj(fi,fo)
        else: shutil.copy(src,dst)
    return dst

def keymap(fa, expect=CHRLEN, tol=0.05):
    keys=list(fa.keys()); probe=keys if len(keys)<=200 else keys[:50]
    lens={k:len(fa[k]) for k in probe}
    def bad(m): return [c for c in expect if m.get(c) not in lens or abs(lens[m[c]]-expect[c])>tol*expect[c]]
    cands=[{f'chr{i}':pat.format(i=i) for i in range(1,13)}
           for pat in ['{i}','chr{i}','chr{i:02d}','Chr{i}','Chr{i:02d}','CHR{i}','chromosome{i}']]
    if len(keys)>=12: cands.append({f'chr{i+1}':keys[i] for i in range(12)})
    for m in cands:
        if not bad(m): return m,[]
    best=min(cands,key=lambda m:len(bad(m))); return best,bad(best)

rows=[]
for r in matched.head(20).itertuples():
    loc=to_local(r.fasta); fa=Fasta(loc); m,_=keymap(fa)
    rows.append({'acc':r.id, **{c:len(fa[m[c]]) for c in CHRLEN if m.get(c) in fa.keys()}})
    fa.close()
    for p in [loc,loc+'.fai']:
        if os.path.exists(p): os.remove(p)
lens=pd.DataFrame(rows).set_index('acc'); lens.to_csv(f'{OUT}/chrom_lengths_check.csv')
print(lens.to_string())
nuni=lens.nunique()
no_drift=bool((nuni==1).all())
print('\nSố chiều dài KHÁC NHAU mỗi NST:'); print(nuni.to_string())
if no_drift:
    print('\nKẾT LUẬN: KHÔNG drift. CHECKPOINT 4: PASS')
else:
    print(f'\nKẾT LUẬN: CÓ drift (lệch tối đa {int((lens.max()-lens.min()).max()):,} bp).')
    print('=> Phần 5 neo homology KHÔNG dùng tọa độ tuyệt đối nên không bị ảnh hưởng. CHECKPOINT 4: PASS')

                   chr1      chr2      chr3      chr4      chr5      chr6      chr7      chr8      chr9     chr10     chr11     chr12
acc                                                                                                                                  
B015           43237252  35907988  36387448  35481652  29939665  31226936  29676088  28422991  22996092  23188267  28999967  27511262
B061           43238205  35908608  36388099  35482131  29940078  31226155  29676720  28423597  22996370  23189014  29000999  27512800
B081           43238579  35910261  36387457  35482695  29939479  31226143  29676345  28423316  22996051  23189252  29001255  27514424
B094           43241178  35912503  36392539  35481621  29941026  31226744  29677782  28424080  22998019  23190183  29001324  27515414
B149           43238150  35905960  36388172  35480709  29940666  31224424  29675117  28423769  22993918  23188984  28998888  27515162
B157           43240844  35909777  36389255  35481887  2993975

## 5. Neo homology 9 gene vào từng mẫu → trích locus
Với mỗi mẫu: index genome bằng mappy, map từng query. Lấy các hit tốt trên **một contig + một chiều**, gộp thành span, nới `FLANK` hai phía → trích trình tự locus **theo chính mẫu đó** (kèm promoter). Đây là bước thay thế cắt-theo-tọa-độ, tránh trượt tọa độ.

QC mỗi hit: identity, coverage, %N, số hit phụ (paralog).

In [ ]:
import glob
done = glob.glob(f'{OUT_SEQ}/*.npz')
print(f'Đã trích: {len(done)}/327 mẫu')

Đã trích: 30/327 mẫu


In [ ]:
# === NẠP LẠI PA_MAT từ file đã lưu (không align lại) ===
import pandas as pd, os

f = f'{OUT}/presence_absence_groupB.csv'
assert os.path.exists(f), f'Chưa có file {f} — cần chạy cell presence/absence trước'

pa_df = pd.read_csv(f)
PA_MAT = pa_df.pivot(index='id', columns='locus', values='present').fillna(0).astype(int)

print(f'Đã nạp PA_MAT: {PA_MAT.shape[0]} mẫu × {PA_MAT.shape[1]} locus')
print('Số mẫu CÓ mỗi gen:')
print(PA_MAT.sum().to_string())
print('\nLocus B:', list(PA_MAT.columns))

Đã nạp PA_MAT: 327 mẫu × 4 locus
Số mẫu CÓ mỗi gen:
locus
Xa10      0
Xa23    259
Xa48     50
Xa7       0

Locus B: ['Xa10', 'Xa23', 'Xa48', 'Xa7']


In [ ]:
# =====================================================================
# CELL GỘP (LOW-RAM): chuẩn bị + trích xuất Nhóm A
# Sửa rò rỉ RAM: bỏ splice (nửa index), del + gc mỗi mẫu, theo dõi RAM.
# CHẠY LẠI CẢ CELL sau mỗi lần Colab ngắt — tự resume.
# =====================================================================
import os, glob, time, shutil, gzip, re, gc
import numpy as np, pandas as pd
import mappy as mp
from pyfaidx import Fasta
try: import psutil; HAS_PSUTIL=True
except: HAS_PSUTIL=False

# ---- CONFIG (sửa cho khớp Drive) ----
ROOT      = '/content/drive/MyDrive/Project/Bioinformatics - Thesis'
OUT       = f'{ROOT}/Export'
OUT_SEQ   = f'{OUT}/regions_seq'
TMP       = '/content/work'
FASTA_DIR = '/content/drive/MyDrive/consensus_results'
GENE9_XLS = f'{ROOT}/9_gene_BB_thong_tin_khoa_hoc_va_trinh_tu.xlsx'
FLANK, SEED = 2000, 42
os.makedirs(OUT_SEQ, exist_ok=True); os.makedirs(TMP, exist_ok=True)

def ram():
    return f'{psutil.virtual_memory().used/1e9:.1f}GB' if HAS_PSUTIL else '?'

def to_local(drive_path):
    base=os.path.basename(drive_path)
    if base.endswith('.gz'):
        tmp_gz=f'{TMP}/{base}'; local=f'{TMP}/{base[:-3]}'
        if not os.path.exists(local):
            shutil.copy(drive_path, tmp_gz)
            with gzip.open(tmp_gz,'rb') as fi, open(local,'wb') as fo:
                shutil.copyfileobj(fi, fo)
            os.remove(tmp_gz)
    else:
        local=f'{TMP}/{base}'
        if not os.path.exists(local): shutil.copy(drive_path, local)
    return local

# ---- nạp matched ----
if 'matched' not in globals():
    mc=f'{OUT}/matched.csv'
    if os.path.exists(mc):
        matched=pd.read_csv(mc)
    else:
        pats=['*.fa','*.fasta','*.fna','*.fa.gz','*.fasta.gz','*.fna.gz']
        files=sorted(set(sum([glob.glob(f'{FASTA_DIR}/{p}') for p in pats],[])))
        def stem(p):
            b=os.path.basename(p)
            for e in ['.fasta.gz','.fa.gz','.fna.gz','.fasta','.fa','.fna']:
                if b.endswith(e): return b[:-len(e)]
            return os.path.splitext(b)[0]
        fmap={stem(f):f for f in files}
        pheno=pd.read_csv(f'{ROOT}/pheno.csv')
        idc='id' if 'id' in pheno.columns else pheno.columns[0]
        pheno['id']=pheno[idc].astype(str); pheno['fasta']=pheno['id'].map(fmap)
        matched=pheno.dropna(subset=['fasta']).reset_index(drop=True)
print(f'matched: {len(matched)} mẫu | RAM {ram()}')

# ---- dựng QUERIES Nhóm A ----
if 'QUERIES' not in globals() or 'REG_IDS_A' not in globals():
    GROUP_B_LOCI={'Xa23','Xa48','Xa7','Xa10'}
    g9=pd.read_excel(GENE9_XLS, sheet_name=0, header=4); g9.columns=[str(c).strip() for c in g9.columns]
    COL_GENE='Gene / locus'; COL_TYPE='Loại sequence được cung cấp'; COL_SEQ='Trình tự DNA 5′→3′'
    g9=g9.dropna(subset=[COL_SEQ]).reset_index(drop=True)
    def clean_seq(x): return re.sub(r'\s','',''.join(l for l in str(x).split('\n') if not l.startswith('>'))).upper()
    def region_id(g): return re.sub(r'[^A-Za-z0-9]+','_',str(g)).strip('_')
    QUERIES=[]
    for _,r in g9.iterrows():
        rid=region_id(r[COL_GENE])
        QUERIES.append(dict(rid=rid, seq=clean_seq(r[COL_SEQ]),
                            group=('B' if rid in GROUP_B_LOCI else 'A')))
    REG_IDS_A=[q['rid'] for q in QUERIES if q['group']=='A']
    print(f'Nhóm A: {REG_IDS_A}')

def best_span(aln, q):
    good=[]
    for h in aln.map(q):
        cov=(h.q_en-h.q_st)/max(len(q),1); idt=h.mlen/max(h.blen,1)
        good.append((h.ctg,h.strand,h.r_st,h.r_en,idt,cov,h.mlen))
    if not good: return None
    from collections import defaultdict
    agg=defaultdict(lambda:[1e18,-1,0.0,0.0,0])
    for ctg,st,rs,re_,idt,cov,ml in good:
        k=(ctg,st); a=agg[k]; a[0]=min(a[0],rs); a[1]=max(a[1],re_); a[2]+=idt*ml; a[3]+=ml; a[4]+=1
    key=max(agg,key=lambda k:agg[k][3]); a=agg[key]
    idt=a[2]/max(a[3],1); cov=min(1.0,a[3]/max(len(q),1))
    if idt<0.80 or cov<0.60: return None
    return (key[0],key[1],int(a[0]),int(a[1]))

Q_A=[qq for qq in QUERIES if qq['rid'] in REG_IDS_A]
t0=time.time(); todo=matched.reset_index(drop=True); n_done=0

for n,acc in enumerate(todo.itertuples(),1):
    done_f=f'{OUT_SEQ}/{acc.id}.npz'
    if os.path.exists(done_f): n_done+=1; continue
    loc=to_local(acc.fasta)
    aln=None; fa=None
    try:
        aln=mp.Aligner(loc, preset='asm20')   # CHỈ asm20 -> nửa RAM (bỏ splice)
        fa=Fasta(loc, rebuild=False); seqs={}
        for q in Q_A:
            sp=best_span(aln, q['seq'])
            if sp is None: continue
            ctg,strand,rs,re_=sp
            L=len(fa[ctg]); s2=max(0,rs-FLANK); e2=min(L,re_+FLANK)
            seqs[q['rid']]=str(fa[ctg][s2:e2]).upper()
        np.savez_compressed(done_f, **seqs)
    finally:
        # === GIẢI PHÓNG RAM TRIỆT ĐỂ ===
        if fa is not None:
            try: fa.close()
            except: pass
        del aln, fa                            # xóa index + fasta handle
        gc.collect()                           # gom rác NGAY
        for p in [loc, loc+'.fai']:            # xóa file giải nén khỏi ổ
            if os.path.exists(p): os.remove(p)
    if n%10==0:
        el=time.time()-t0; dn=max(1,n-n_done)
        print(f'  {n}/{len(todo)} (đã có {n_done}) | {el/dn:.1f}s/mẫu | RAM {ram()}', flush=True)

done_total=len(glob.glob(f'{OUT_SEQ}/*.npz'))
print(f'\n=== Phiên này xong. Đã trích: {done_total}/327 | RAM {ram()} ===')
print('>>> Ngắt thì CHẠY LẠI CẢ CELL — tự resume <<<' if done_total<327 else '>>> ĐỦ 327. Sang lọc %N + embedding <<<')

matched: 327 mẫu | RAM 2.2GB
  40/327 (đã có 30) | 20.7s/mẫu | RAM 3.1GB
  50/327 (đã có 30) | 20.9s/mẫu | RAM 3.1GB
  60/327 (đã có 30) | 21.1s/mẫu | RAM 3.1GB
  70/327 (đã có 30) | 21.3s/mẫu | RAM 3.1GB
  80/327 (đã có 30) | 21.4s/mẫu | RAM 3.0GB
  90/327 (đã có 30) | 21.5s/mẫu | RAM 2.8GB
  100/327 (đã có 30) | 21.6s/mẫu | RAM 2.9GB
  110/327 (đã có 30) | 21.6s/mẫu | RAM 3.1GB
  120/327 (đã có 30) | 21.6s/mẫu | RAM 2.8GB
  130/327 (đã có 30) | 21.6s/mẫu | RAM 3.1GB
  140/327 (đã có 30) | 21.7s/mẫu | RAM 3.0GB
  150/327 (đã có 30) | 21.7s/mẫu | RAM 3.0GB
  160/327 (đã có 30) | 21.8s/mẫu | RAM 3.0GB
  170/327 (đã có 30) | 21.7s/mẫu | RAM 2.9GB
  180/327 (đã có 30) | 21.7s/mẫu | RAM 3.1GB
  190/327 (đã có 30) | 21.7s/mẫu | RAM 2.8GB
  200/327 (đã có 30) | 21.6s/mẫu | RAM 2.9GB
  210/327 (đã có 30) | 21.6s/mẫu | RAM 3.0GB
  220/327 (đã có 30) | 21.6s/mẫu | RAM 3.1GB
  230/327 (đã có 30) | 21.6s/mẫu | RAM 3.1GB
  240/327 (đã có 30) | 21.7s/mẫu | RAM 3.1GB
  250/327 (đã có 30) | 21.9s/mẫu

Kiểm tra align tìm đúng gene, identity phải cao (>0.9) và coverage tốt

In [ ]:
import os, re, glob
import numpy as np, pandas as pd
import mappy as mp

# ---- config ----
ROOT='/content/drive/MyDrive/Project/Bioinformatics - Thesis'
OUT=f'{ROOT}/Export'; TMP='/content/work'
GENE9_XLS=f'{ROOT}/Bacterial Leaf Data/9_gene_BB_thong_tin_khoa_hoc_va_trinh_tu.xlsx'
os.makedirs(TMP, exist_ok=True)

# ---- to_local ----
import shutil, gzip
def to_local(p):
    b=os.path.basename(p)
    if b.endswith('.gz'):
        tg=f'{TMP}/{b}'; loc=f'{TMP}/{b[:-3]}'
        if not os.path.exists(loc):
            shutil.copy(p,tg)
            with gzip.open(tg,'rb') as fi, open(loc,'wb') as fo: shutil.copyfileobj(fi,fo)
            os.remove(tg)
    else:
        loc=f'{TMP}/{b}'
        if not os.path.exists(loc): shutil.copy(p,loc)
    return loc

# ---- matched ----
matched=pd.read_csv(f'{OUT}/matched.csv')

# ---- QUERIES (Nhóm A) ----
GROUP_B_LOCI={'Xa23','Xa48','Xa7','Xa10'}
g9=pd.read_excel(GENE9_XLS, sheet_name=0, header=4); g9.columns=[str(c).strip() for c in g9.columns]
COL_GENE='Gene / locus'; COL_SEQ='Trình tự DNA 5′→3′'
g9=g9.dropna(subset=[COL_SEQ]).reset_index(drop=True)
def clean_seq(x): return re.sub(r'\s','',''.join(l for l in str(x).split('\n') if not l.startswith('>'))).upper()
def region_id(g): return re.sub(r'[^A-Za-z0-9]+','_',str(g)).strip('_')
QUERIES=[]
for _,r in g9.iterrows():
    rid=region_id(r[COL_GENE])
    QUERIES.append(dict(rid=rid, seq=clean_seq(r[COL_SEQ]), group=('B' if rid in GROUP_B_LOCI else 'A')))
REG_IDS_A=[q['rid'] for q in QUERIES if q['group']=='A']
print('Dựng lại xong. REG_IDS_A:', REG_IDS_A)

# ---- KIỂM 3 mẫu ----
Q_A=[q for q in QUERIES if q['rid'] in REG_IDS_A]
for acc in list(matched.itertuples())[:3]:
    loc=to_local(acc.fasta); aln=mp.Aligner(loc,preset='asm20')
    print(f'\n=== {acc.id} ===')
    for q in Q_A:
        bi,bc=0,0
        for h in aln.map(q['seq']):
            cov=(h.q_en-h.q_st)/max(len(q['seq']),1); idt=h.mlen/max(h.blen,1)
            if cov*idt>bc*bi: bi,bc=idt,cov
        print(f"  {q['rid']:14s} identity={bi:.3f} coverage={bc:.3f}")
    del aln
    for p in [loc,loc+'.fai']:
        if os.path.exists(p): os.remove(p)

Dựng lại xong. REG_IDS_A: ['Xa1', 'Xa21', 'Xa3_Xa26', 'xa5', 'xa13_SWEET11', 'xa25_SWEET13', 'Xa27']

=== B015 ===
  Xa1            identity=0.999 coverage=0.450
  Xa21           identity=0.957 coverage=0.978
  Xa3_Xa26       identity=0.861 coverage=0.161
  xa5            identity=0.989 coverage=1.000
  xa13_SWEET11   identity=0.940 coverage=1.000
  xa25_SWEET13   identity=0.969 coverage=0.995
  Xa27           identity=0.938 coverage=0.456

=== B061 ===
  Xa1            identity=0.999 coverage=0.450
  Xa21           identity=0.957 coverage=0.978
  Xa3_Xa26       identity=0.864 coverage=0.161
  xa5            identity=0.988 coverage=1.000
  xa13_SWEET11   identity=0.940 coverage=1.000
  xa25_SWEET13   identity=0.968 coverage=0.995
  Xa27           identity=0.938 coverage=0.456

=== B081 ===
  Xa1            identity=0.998 coverage=0.450
  Xa21           identity=0.959 coverage=0.978
  Xa3_Xa26       identity=0.983 coverage=0.715
  xa5            identity=0.989 coverage=1.000
  xa13_SWEE

## 6. Lọc %N
Locus có `%N > MAX_PCTN_GENE` bị loại khỏi embedding (tránh học nhiễu chất lượng lắp ráp). Gene chỉ giữ nếu đủ số mẫu sạch.

In [ ]:
# === Dựng lại rep từ regions_seq đã trích (không chạy lại trích xuất) ===
import glob, os, numpy as np, pandas as pd

rows=[]
for f in glob.glob(f'{OUT_SEQ}/*.npz'):
    acc=os.path.basename(f)[:-4]           # bỏ .npz
    z=np.load(f, allow_pickle=True)
    for rid in REG_IDS_A:
        if rid in z.files:
            s=str(z[rid])
            pctN=round(100*s.count('N')/max(len(s),1),2)
            rows.append(dict(acc=acc, rid=rid, status='OK', length=len(s),
                             pctN=pctN, identity=np.nan, coverage=np.nan))
        else:
            rows.append(dict(acc=acc, rid=rid, status='NO_HIT'))

rep=pd.DataFrame(rows)
rep.to_csv(f'{OUT}/extract_report.csv', index=False)
print(f'Dựng lại rep: {len(rep)} dòng | {rep.acc.nunique()} mẫu')
print(rep.status.value_counts().to_string())

Dựng lại rep: 2289 dòng | 327 mẫu
status
OK        1905
NO_HIT     384


In [ ]:
ok_rep=rep[rep.status=='OK']
qual=(ok_rep.groupby('rid')
        .agg(pctN=('pctN','mean'), n_ok=('acc','nunique'),
             idt=('identity','mean'), cov=('coverage','mean')).reset_index())
qual['clean']=(qual.pctN<=MAX_PCTN_GENE)&(qual.n_ok>=0.85*len(matched))
print(qual.sort_values('pctN').to_string(index=False))
GENE_OK=qual[qual.clean].rid.tolist()
print(f'\nGene sạch dùng cho embedding: {len(GENE_OK)} -> {GENE_OK}')
qual.to_csv(f'{OUT}/gene_quality.csv', index=False)
# mask %N theo từng (mẫu, gene) để loại locus bẩn ở mức mẫu khi build ma trận
pctN_tab = ok_rep.pivot_table(index='acc', columns='rid', values='pctN')
print('CHECKPOINT 6:', 'PASS' if len(GENE_OK)>=4 else 'FAIL -> nới MAX_PCTN')

         rid      pctN  n_ok  idt  cov  clean
         xa5  0.370061   327  NaN  NaN   True
xa13_SWEET11  0.534495   327  NaN  NaN   True
xa25_SWEET13  3.763578   327  NaN  NaN   True
        Xa21  7.620742   310  NaN  NaN   True
         Xa1 20.352047   254  NaN  NaN  False
    Xa3_Xa26 45.977630   173  NaN  NaN  False
        Xa27 51.047968   187  NaN  NaN  False

Gene sạch dùng cho embedding: 4 -> ['Xa21', 'xa13_SWEET11', 'xa25_SWEET13', 'xa5']
CHECKPOINT 6: PASS


## 7. Embedding DNABERT theo locus
Frozen model. Mỗi locus: cửa sổ trượt 1000/500, bỏ cửa sổ nhiều N, masked-mean-pool token → vector 768 (hoặc mean+max = 1536). Có **checkpoint** để chạy tiếp nếu Colab ngắt phiên.

In [ ]:
import torch, sys
from transformers import AutoTokenizer, AutoModel, BertConfig
dev='cuda' if torch.cuda.is_available() else 'cpu'

if EMB_MODEL=='plant-dnabert':
    MODEL='zhangtaolab/plant-dnabert-BPE'; REV='10e9c3c73f12598f46e3007ed11be483169d5dd1'
    tok=AutoTokenizer.from_pretrained(MODEL, revision=REV, trust_remote_code=True)
    mdl=AutoModel.from_pretrained(MODEL, revision=REV, trust_remote_code=True).to(dev).eval()
else:  # dnabert2 (cần vá triton/flash-attn như bản Windows)
    import transformers.dynamic_module_utils as dmu
    if not hasattr(dmu,'_orig'): dmu._orig=dmu.check_imports
    def _patched(fn):
        try: return dmu._orig(fn)
        except ImportError as e:
            if 'triton' in str(e).lower(): return dmu.get_relative_imports(fn)
            raise
    dmu.check_imports=_patched
    MODEL='zhihan1996/DNABERT-2-117M'
    tok=AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
    cfg=BertConfig.from_pretrained(MODEL)
    if getattr(cfg,'pad_token_id',None) is None: cfg.pad_token_id=tok.pad_token_id or 3
    mdl=AutoModel.from_pretrained(MODEL, config=cfg, trust_remote_code=True).to(dev).eval()
    for nm,m in list(sys.modules.items()):
        if 'bert_layers' in nm: m.flash_attn_qkvpacked_func=None
print('embedding model:', MODEL, '| device:', dev)

WINDOW,STRIDE,MAX_N_WIN,MIN_WIN,BATCH = 1000,500,0.20,3,128
@torch.no_grad()
def embed_region(seq):
    wins=[seq[i:i+WINDOW] for i in range(0,max(1,len(seq)-WINDOW+1),STRIDE)]
    wins=[w for w in wins if len(w)>=200 and w.count('N')/len(w)<=MAX_N_WIN]
    if len(wins)<MIN_WIN: return None
    out=[]
    for i in range(0,len(wins),BATCH):
        enc=tok(wins[i:i+BATCH],return_tensors='pt',padding=True,truncation=True,max_length=512)
        ids,msk=enc['input_ids'].to(dev),enc['attention_mask'].to(dev)
        with torch.autocast('cuda',dtype=torch.float16):
            h=mdl(ids,attention_mask=msk)[0]
        h=h.float(); m=msk.unsqueeze(-1).float()
        out.append(((h*m).sum(1)/m.sum(1).clamp(min=1)).cpu())
    W=torch.cat(out,0).numpy()
    return W.mean(0).astype('float32') if POOL=='mean' else np.concatenate([W.mean(0),W.max(0)]).astype('float32')

Some weights of BertModel were not initialized from the model checkpoint at zhangtaolab/plant-dnabert-BPE and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


embedding model: zhangtaolab/plant-dnabert-BPE | device: cuda


In [ ]:
import os
import time
import numpy as np
import pandas as pd

# === THÊM DÒNG NÀY: danh sách gene để embedding = 4 gene sạch + Xa1 (cứu bằng C) ===
GENE_EMBED = GENE_OK + ['Xa1']
print('Gene sẽ embedding:', GENE_EMBED)

# chạy embedding (có checkpoint)
E={}; PART=f'{OUT}/embeddings_partial.npz'
if os.path.exists(PART):
    zz=np.load(PART)
    for k in zz.files:
        a,r=k.split('|',1); E.setdefault(a,{})[r]=zz[k]
    print('đã có', len(E), 'mẫu — chạy tiếp')

t0=time.time(); n_new=0
for n,acc in enumerate(matched.itertuples(),1):
    f=f'{OUT_SEQ}/{acc.id}.npz'
    if not os.path.exists(f): continue
    # === SỬA: kiểm tra theo GENE_EMBED, nhưng Xa1 có thể thiếu (không bắt buộc) ===
    need = [r for r in GENE_EMBED if r != 'Xa1']   # 4 gene sạch bắt buộc
    if acc.id in E and all(r in E[acc.id] for r in need): continue
    z=np.load(f, allow_pickle=True); row=E.get(acc.id,{})
    for rid in GENE_EMBED:                          # === SỬA: GENE_OK -> GENE_EMBED ===
        if rid in row: continue
        if rid not in z.files: continue
        # bỏ locus bẩn ở mức mẫu (áp cho Xa1: mẫu %N cao -> không embedding -> C sẽ imputation)
        if rid in pctN_tab.columns and acc.id in pctN_tab.index:
            v=pctN_tab.loc[acc.id, rid]
            if pd.notna(v) and v>MAX_PCTN_SAMPLE: continue
        e=embed_region(str(z[rid]))
        if e is not None: row[rid]=e
    E[acc.id]=row; n_new+=1
    if n_new%5==0:
        el=time.time()-t0
        print(f'  {n}/{len(matched)} | {el/n_new:.1f}s/mẫu | còn ~{(len(matched)-n)*el/n_new/60:.0f} ph', flush=True)
        np.savez_compressed(PART, **{f'{a}|{r}':v for a,dd in E.items() for r,v in dd.items()})

np.savez_compressed(f'{OUT}/embeddings.npz', **{f'{a}|{r}':v for a,dd in E.items() for r,v in dd.items()})
print(f'\nXong {len(E)} mẫu trong {(time.time()-t0)/60:.1f} phút')
# thống kê Xa1 embedding được ở bao nhiêu mẫu
n_xa1 = sum(1 for a in E if 'Xa1' in E[a])
print(f'Xa1 embedding được ở {n_xa1} mẫu (còn lại sẽ imputation ở cell ghép C)')
print('CHECKPOINT 7:', 'PASS' if len(E)>=250 else 'FAIL')

Gene sẽ embedding: ['Xa21', 'xa13_SWEET11', 'xa25_SWEET13', 'xa5', 'Xa1']
  5/327 | 0.6s/mẫu | còn ~3 ph
  10/327 | 0.5s/mẫu | còn ~3 ph
  15/327 | 0.5s/mẫu | còn ~3 ph
  20/327 | 0.5s/mẫu | còn ~3 ph
  25/327 | 0.5s/mẫu | còn ~3 ph
  30/327 | 0.5s/mẫu | còn ~2 ph
  35/327 | 0.5s/mẫu | còn ~2 ph
  40/327 | 0.5s/mẫu | còn ~2 ph
  45/327 | 0.4s/mẫu | còn ~2 ph
  50/327 | 0.4s/mẫu | còn ~2 ph
  55/327 | 0.4s/mẫu | còn ~2 ph
  60/327 | 0.4s/mẫu | còn ~2 ph
  65/327 | 0.4s/mẫu | còn ~2 ph
  70/327 | 0.4s/mẫu | còn ~2 ph
  75/327 | 0.4s/mẫu | còn ~2 ph
  80/327 | 0.4s/mẫu | còn ~2 ph
  85/327 | 0.4s/mẫu | còn ~2 ph
  90/327 | 0.4s/mẫu | còn ~2 ph
  95/327 | 0.4s/mẫu | còn ~1 ph
  100/327 | 0.4s/mẫu | còn ~1 ph
  105/327 | 0.4s/mẫu | còn ~1 ph
  110/327 | 0.4s/mẫu | còn ~1 ph
  115/327 | 0.4s/mẫu | còn ~1 ph
  120/327 | 0.4s/mẫu | còn ~1 ph
  125/327 | 0.4s/mẫu | còn ~1 ph
  130/327 | 0.4s/mẫu | còn ~1 ph
  135/327 | 0.4s/mẫu | còn ~1 ph
  140/327 | 0.4s/mẫu | còn ~1 ph
  145/327 | 0.4s/mẫu |

In [ ]:
print('POOL =', POOL, '| hidden_size =', mdl.config.hidden_size)

POOL = mean | hidden_size = 768


In [ ]:
print('POOL =', POOL)                    # phải là 'mean'
print('GENE_OK =', GENE_OK)              # 4 gene sạch
GENE_EMBED = GENE_OK + ['Xa1']           # thêm Xa1 cho phương pháp C
print('GENE_EMBED =', GENE_EMBED)        # 5 gene

POOL = mean
GENE_OK = ['Xa21', 'xa13_SWEET11', 'xa25_SWEET13', 'xa5']
GENE_EMBED = ['Xa21', 'xa13_SWEET11', 'xa25_SWEET13', 'xa5', 'Xa1']


In [ ]:
a0=next(iter(E)); r0=list(E[a0].keys())[0]
print('chiều 1 gene:', E[a0][r0].shape)   # LAN NAY phải là (768,)

chiều 1 gene: (768,)


## 8. Ghép ma trận đặc trưng
`X_emb` = nối embedding của các gene sạch theo thứ tự cố định. Chỉ giữ mẫu có đủ toàn bộ `GENE_OK` (đơn giản, chặt). Có thể đổi sang zero-vector + missing-indicator nếu muốn giữ nhiều mẫu hơn.

In [ ]:
# =====================================================================
# GHÉP MA TRẬN ĐẶC TRƯNG CUỐI — phương pháp C (mask Xa1 theo mẫu)
# Chạy SAU khi có embedding (dict E) + PA_MAT (presence/absence Nhóm B)
#
# Cấu trúc đặc trưng cuối:
#   [embedding 5 gene: xa5, xa13, xa25, Xa21, Xa1(masked)]
#   + [cột chỉ báo Xa1_present]
#   + [presence/absence: Xa23, Xa48]
# =====================================================================
import numpy as np, pandas as pd

# ---- THAM SỐ phương pháp C ----
GENE_EMB    = ['xa5', 'xa13_SWEET11', 'xa25_SWEET13', 'Xa21', 'Xa1']  # 5 gene embedding
GENE_MASK   = 'Xa1'          # gene áp dụng mask theo mẫu
PCTN_SAMPLE = 10.0           # mẫu có %N(Xa1) > 10% -> mask (coi là missing)

# ---- nạp embedding nếu chưa có ----
if 'E' not in globals() or not E:
    ze=np.load(f'{OUT}/embeddings.npz'); E={}
    for k in ze.files:
        a,r=k.split('|',1); E.setdefault(a,{})[r]=ze[k]
    print(f'Đã nạp embedding: {len(E)} accession')

# ---- KIỂM TRA tên gene khớp (quan trọng!) ----
sample_keys = list(next(iter(E.values())).keys())
print('Tên gene trong embedding:', sample_keys)
missing = [g for g in GENE_EMB if g not in sample_keys and g != GENE_MASK]
if missing:
    print(f'⚠️ CẢNH BÁO: các gene sau trong GENE_EMB không có trong embedding: {missing}')
    print('   -> Sửa GENE_EMB cho khớp tên thật ở trên rồi chạy lại.')

# ---- nạp PA_MAT nếu chưa có ----
if 'PA_MAT' not in globals():
    pa=pd.read_csv(f'{OUT}/presence_absence_groupB.csv')
    PA_MAT=pa.pivot(index='id',columns='locus',values='present').fillna(0).astype(int)
    print(f'Đã nạp PA_MAT: {PA_MAT.shape}')

# lọc PA_MAT còn 2 gene thật (bỏ Xa7, Xa10 - cột chết)
PA_MAT = PA_MAT[[c for c in ['Xa23','Xa48'] if c in PA_MAT.columns]]
print('PA_MAT sau lọc:', list(PA_MAT.columns))

# ---- nạp bảng %N theo mẫu×gene ----
if 'pctN_tab' not in globals():
    rep=pd.read_csv(f'{OUT}/extract_report.csv')
    ok=rep[rep.status=='OK']
    pctN_tab=ok.pivot_table(index='acc', columns='rid', values='pctN')
    print('Đã dựng pctN_tab')

# =====================================================================
# XÁC ĐỊNH mẫu dùng được + tính vector Xa1 trung bình (imputation)
# =====================================================================
GENE_REQ = [g for g in GENE_EMB if g != GENE_MASK]   # 4 gene bắt buộc có
emb_dim=None
for a in E:
    for g in GENE_REQ:
        if g in E[a]: emb_dim=len(E[a][g]); break
    if emb_dim: break
print(f'Chiều embedding mỗi gene: {emb_dim}')

xa1_clean_vecs=[]
for a in E:
    if GENE_MASK in E[a]:
        pn = pctN_tab.loc[a, GENE_MASK] if (a in pctN_tab.index and GENE_MASK in pctN_tab.columns) else 100
        if pd.notna(pn) and pn <= PCTN_SAMPLE:
            xa1_clean_vecs.append(E[a][GENE_MASK])
xa1_mean = np.mean(xa1_clean_vecs, axis=0) if xa1_clean_vecs else np.zeros(emb_dim)
print(f'Xa1: {len(xa1_clean_vecs)} mẫu sạch (%N≤{PCTN_SAMPLE}%) dùng tính vector trung bình')

# =====================================================================
# DỰNG MA TRẬN
# =====================================================================
rows_X=[]; rows_id=[]; xa1_present_flags=[]; n_masked=0
for a in df_snp.id:
    if a not in E: continue
    if not all(g in E[a] for g in GENE_REQ): continue   # bắt buộc đủ 4 gene sạch
    vecs=[]
    for g in GENE_EMB:
        if g == GENE_MASK:
            pn = pctN_tab.loc[a, GENE_MASK] if (a in pctN_tab.index and GENE_MASK in pctN_tab.columns) else 100
            if (GENE_MASK in E[a]) and pd.notna(pn) and pn <= PCTN_SAMPLE:
                vecs.append(E[a][GENE_MASK]); flag=1        # sạch -> dùng thật
            else:
                vecs.append(xa1_mean); flag=0; n_masked+=1  # bẩn/thiếu -> imputation
            xa1_present_flags.append(flag)
        else:
            vecs.append(E[a][g])
    rows_X.append(np.concatenate(vecs)); rows_id.append(a)

X_emb_part = np.vstack(rows_X)
ids_keep   = rows_id
xa1_flag   = np.array(xa1_present_flags).reshape(-1,1)
print(f'\nEmbedding part: {X_emb_part.shape} | {len(ids_keep)} mẫu')
print(f'Xa1 bị mask (imputation): {n_masked}/{len(ids_keep)} mẫu')

# ---- ghép presence/absence (Xa23, Xa48) theo đúng thứ tự ids_keep ----
PA_aligned = PA_MAT.reindex(ids_keep).fillna(0).astype(int).values
print(f'PA part: {PA_aligned.shape} ({list(PA_MAT.columns)})')

# ---- GHÉP CUỐI ----
X_full = np.hstack([X_emb_part, xa1_flag, PA_aligned])
print(f'\n=== MA TRẬN ĐẶC TRƯNG CUỐI: {X_full.shape} ===')
print(f'  = {X_emb_part.shape[1]} (embedding 5 gene)'
      f' + 1 (Xa1_present) + {PA_aligned.shape[1]} (presence/absence)')

# ---- căn nhãn + nhóm ----
idx_map = {a:i for i,a in enumerate(df_snp.id)}
keep_rows = [idx_map[a] for a in ids_keep]
df_keep = df_snp.iloc[keep_rows].reset_index(drop=True)
g_full  = df_keep.subgroup.values

np.savez_compressed(f'{OUT}/X_full_14locus.npz', X_full=X_full, ids=np.array(ids_keep))
df_keep.to_pickle(f'{OUT}/df_keep_14locus.pkl')
print(f'\nĐã lưu X_full_14locus.npz + df_keep_14locus.pkl')
print(f'Dùng X_full, df_keep, g_full cho train.')
print('CHECKPOINT GHÉP:', 'PASS' if X_full.shape[0]>=200 else 'FAIL')

Tên gene trong embedding: ['Xa21', 'xa13_SWEET11', 'xa25_SWEET13', 'xa5', 'Xa1']
PA_MAT sau lọc: ['Xa23', 'Xa48']
Chiều embedding mỗi gene: 768
Xa1: 135 mẫu sạch (%N≤10.0%) dùng tính vector trung bình

Embedding part: (303, 3840) | 303 mẫu
Xa1 bị mask (imputation): 172/303 mẫu
PA part: (303, 2) (['Xa23', 'Xa48'])

=== MA TRẬN ĐẶC TRƯNG CUỐI: (303, 3843) ===
  = 3840 (embedding 5 gene) + 1 (Xa1_present) + 2 (presence/absence)

Đã lưu X_full_14locus.npz + df_keep_14locus.pkl
Dùng X_full, df_keep, g_full cho train.
CHECKPOINT GHÉP: PASS


In [ ]:
print('X_full:', X_full.shape)
print('df_keep:', df_keep.shape, '| g_full:', len(g_full))
print('Khớp:', X_full.shape[0]==len(df_keep)==len(g_full))

X_full: (303, 3843)
df_keep: (303, 22) | g_full: 303
Khớp: True


## 9. Module đánh giá chuẩn và Định nghĩa các biến train
Phân loại nhị phân từng chủng (trừ chủng "V" vì mất cân bằng). Hai lược đồ CV: **randomCV** (StratifiedKFold) và **GroupKFold theo subgroup** (chặn rò rỉ cấu trúc quần thể)

#Trước khi train model, nạp lại các biến và dataset đã lưu khi Colab break connection
Đây là phần cell cần phải chạy nếu colab ngắt kết nối

In [ ]:
import numpy as np, pandas as pd

# đường dẫn (sửa nếu khác)
ROOT = '/content/drive/MyDrive/Project/Bioinformatics - Thesis'
OUT  = f'{ROOT}/Export'
SEED = 42

# nạp X_full + df_keep từ file đã lưu
z = np.load(f'{OUT}/X_full_14locus.npz', allow_pickle=True)
X_full = z['X_full']
df_keep = pd.read_pickle(f'{OUT}/df_keep_14locus.pkl')
g_full = df_keep.subgroup.values

print('X_full:', X_full.shape)
print('df_keep:', df_keep.shape, '| g_full:', len(g_full))
print('Khớp:', X_full.shape[0]==len(df_keep)==len(g_full))

X_full: (303, 3843)
df_keep: (303, 22) | g_full: 303
Khớp: True


In [ ]:
# =====================================================================
# CELL NỀN — chạy TRƯỚC tất cả cell model. Định nghĩa hàm dùng chung.
# 80/20 train/test + 5-fold CV trên train (random + GroupKFold) + test độc lập
# =====================================================================
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef)

TEST_SIZE = 0.20
BIN = [s for s in ['C4','C5','P9a'] if s+'_bin' in df_keep.columns]
print(f'X_full: {X_full.shape} | chủng phân loại: {BIN}')

def _metrics(yt, prob, thr=0.5):
    yp=(prob>=thr).astype(int)
    d=dict(Accuracy=accuracy_score(yt,yp),Precision=precision_score(yt,yp,zero_division=0),
           Recall=recall_score(yt,yp,zero_division=0),F1=f1_score(yt,yp,zero_division=0),
           MCC=matthews_corrcoef(yt,yp))
    try: d['AUC']=roc_auc_score(yt,prob)
    except: d['AUC']=np.nan
    return d

def get_split(strain):
    """80/20 stratified theo nhãn chủng. Trả về chỉ số train/test (cố định theo SEED)."""
    y=df_keep[strain+'_bin'].values.astype(int); idx=np.arange(len(y))
    tr,te=train_test_split(idx,test_size=TEST_SIZE,stratify=y,random_state=SEED)
    return tr,te,y

def eval_sklearn(name, make_model):
    """Model ML (tất định): chạy 1 lần. CV random+group trên train + test độc lập."""
    rows=[]
    for s in BIN:
        tr,te,y=get_split(s)
        Xtr,Xte=X_full[tr],X_full[te]; ytr,yte=y[tr],y[te]; gtr=g_full[tr]
        for cvname,cv,grp in [('CV_random',StratifiedKFold(5,shuffle=True,random_state=SEED),None),
                              ('CV_group',GroupKFold(5),gtr)]:
            acc=[]
            for a,b in cv.split(Xtr,ytr,grp):
                if len(np.unique(ytr[b]))<2: continue
                sc=StandardScaler().fit(Xtr[a])
                mm=make_model(); mm.fit(sc.transform(Xtr[a]),ytr[a])
                prob=mm.predict_proba(sc.transform(Xtr[b]))[:,1] if hasattr(mm,'predict_proba') else mm.predict(sc.transform(Xtr[b]))
                acc.append(_metrics(ytr[b],prob))
            d=pd.DataFrame(acc).mean().to_dict()
            r={'model':name,'strain':s,'eval':cvname}
            for k in ['Accuracy','Precision','Recall','F1','AUC','MCC']: r[k]=round(d.get(k,np.nan),3)
            rows.append(r)
        # test độc lập
        sc=StandardScaler().fit(Xtr); mm=make_model(); mm.fit(sc.transform(Xtr),ytr)
        prob=mm.predict_proba(sc.transform(Xte))[:,1] if hasattr(mm,'predict_proba') else mm.predict(sc.transform(Xte))
        d=_metrics(yte,prob); r={'model':name,'strain':s,'eval':'independent_test'}
        for k in ['Accuracy','Precision','Recall','F1','AUC','MCC']: r[k]=round(d.get(k,np.nan),3)
        rows.append(r)
    df=pd.DataFrame(rows); df.to_csv(f'{OUT}/train_{name}.csv',index=False)
    print(df.to_string(index=False)); return df

print('Cell nền sẵn sàng: get_split, eval_sklearn, _metrics')

X_full: (303, 3843) | chủng phân loại: ['C4', 'C5', 'P9a']
Cell nền sẵn sàng: get_split, eval_sklearn, _metrics


## 10. Machine learning trên Embedding

10.1: Logistic Regression (LogReg)

In [ ]:
from sklearn.linear_model import LogisticRegression
eval_sklearn('LogReg', lambda: LogisticRegression(
    C=0.05, max_iter=3000, class_weight='balanced', random_state=SEED))

 model strain             eval  Accuracy  Precision  Recall    F1   AUC   MCC
LogReg     C4        CV_random     0.645      0.630   0.625 0.625 0.710 0.290
LogReg     C4         CV_group     0.565      0.543   0.561 0.536 0.594 0.113
LogReg     C4 independent_test     0.557      0.543   0.633 0.585 0.661 0.118
LogReg     C5        CV_random     0.761      0.343   0.356 0.335 0.743 0.202
LogReg     C5         CV_group     0.734      0.239   0.337 0.267 0.595 0.089
LogReg     C5 independent_test     0.754      0.357   0.455 0.400 0.744 0.251
LogReg    P9a        CV_random     0.673      0.681   0.650 0.664 0.697 0.348
LogReg    P9a         CV_group     0.679      0.684   0.678 0.669 0.659 0.271
LogReg    P9a independent_test     0.738      0.750   0.700 0.724 0.770 0.476


,model,strain,eval,Accuracy,Precision,Recall,F1,AUC,MCC
0,LogReg,C4,CV_random,0.645,0.630,0.625,0.625,0.710,0.290
1,LogReg,C4,CV_group,0.565,0.543,0.561,0.536,0.594,0.113
2,LogReg,C4,independent_test,0.557,0.543,0.633,0.585,0.661,0.118
3,LogReg,C5,CV_random,0.761,0.343,0.356,0.335,0.743,0.202
4,LogReg,C5,CV_group,0.734,0.239,0.337,0.267,0.595,0.089
5,LogReg,C5,independent_test,0.754,0.357,0.455,0.400,0.744,0.251
6,LogReg,P9a,CV_random,0.673,0.681,0.650,0.664,0.697,0.348
7,LogReg,P9a,CV_group,0.679,0.684,0.678,0.669,0.659,0.271
8,LogReg,P9a,independent_test,0.738,0.750,0.700,0.724,0.770,0.476


10.2: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
eval_sklearn('RandomForest', lambda: RandomForestClassifier(
    n_estimators=300, max_depth=15, max_features='sqrt', min_samples_leaf=5,
    class_weight='balanced', n_jobs=-1, random_state=SEED))

       model strain             eval  Accuracy  Precision  Recall    F1   AUC   MCC
RandomForest     C4        CV_random     0.632      0.629   0.589 0.606 0.697 0.264
RandomForest     C4         CV_group     0.551      0.535   0.635 0.544 0.584 0.097
RandomForest     C4 independent_test     0.689      0.657   0.767 0.708 0.744 0.384
RandomForest     C5        CV_random     0.777      0.357   0.244 0.283 0.774 0.166
RandomForest     C5         CV_group     0.790      0.253   0.228 0.238 0.746 0.116
RandomForest     C5 independent_test     0.770      0.333   0.273 0.300 0.760 0.166
RandomForest    P9a        CV_random     0.744      0.699   0.850 0.767 0.756 0.500
RandomForest    P9a         CV_group     0.757      0.719   0.855 0.774 0.724 0.440
RandomForest    P9a independent_test     0.721      0.676   0.833 0.746 0.835 0.457


,model,strain,eval,Accuracy,Precision,Recall,F1,AUC,MCC
0,RandomForest,C4,CV_random,0.632,0.629,0.589,0.606,0.697,0.264
1,RandomForest,C4,CV_group,0.551,0.535,0.635,0.544,0.584,0.097
2,RandomForest,C4,independent_test,0.689,0.657,0.767,0.708,0.744,0.384
3,RandomForest,C5,CV_random,0.777,0.357,0.244,0.283,0.774,0.166
4,RandomForest,C5,CV_group,0.790,0.253,0.228,0.238,0.746,0.116
5,RandomForest,C5,independent_test,0.770,0.333,0.273,0.300,0.760,0.166
6,RandomForest,P9a,CV_random,0.744,0.699,0.850,0.767,0.756,0.500
7,RandomForest,P9a,CV_group,0.757,0.719,0.855,0.774,0.724,0.440
8,RandomForest,P9a,independent_test,0.721,0.676,0.833,0.746,0.835,0.457


10.3: XGBoost

In [ ]:
from xgboost import XGBClassifier
# tính scale_pos_weight = số mẫu âm / số mẫu dương, trung bình các chủng
import numpy as np
spw = np.mean([(df_keep[s+'_bin']==0).sum()/max((df_keep[s+'_bin']==1).sum(),1)
               for s in ['C4','C5','P9a']])
print('scale_pos_weight ~', round(spw,2))
eval_sklearn('XGBoost', lambda: XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.5, scale_pos_weight=spw,
    n_jobs=-1, random_state=SEED, verbosity=0, eval_metric='logloss'))

scale_pos_weight ~ 2.16
  model strain             eval  Accuracy  Precision  Recall    F1   AUC   MCC
XGBoost     C4        CV_random     0.641      0.624   0.658 0.638 0.681 0.284
XGBoost     C4         CV_group     0.525      0.515   0.695 0.555 0.602 0.068
XGBoost     C4 independent_test     0.623      0.590   0.767 0.667 0.676 0.261
XGBoost     C5        CV_random     0.785      0.375   0.222 0.271 0.744 0.168
XGBoost     C5         CV_group     0.779      0.287   0.247 0.261 0.712 0.124
XGBoost     C5 independent_test     0.803      0.429   0.273 0.333 0.735 0.232
XGBoost    P9a        CV_random     0.739      0.695   0.850 0.764 0.741 0.494
XGBoost    P9a         CV_group     0.740      0.700   0.851 0.756 0.713 0.419
XGBoost    P9a independent_test     0.770      0.750   0.800 0.774 0.858 0.543


,model,strain,eval,Accuracy,Precision,Recall,F1,AUC,MCC
0,XGBoost,C4,CV_random,0.641,0.624,0.658,0.638,0.681,0.284
1,XGBoost,C4,CV_group,0.525,0.515,0.695,0.555,0.602,0.068
2,XGBoost,C4,independent_test,0.623,0.590,0.767,0.667,0.676,0.261
3,XGBoost,C5,CV_random,0.785,0.375,0.222,0.271,0.744,0.168
4,XGBoost,C5,CV_group,0.779,0.287,0.247,0.261,0.712,0.124
5,XGBoost,C5,independent_test,0.803,0.429,0.273,0.333,0.735,0.232
6,XGBoost,P9a,CV_random,0.739,0.695,0.850,0.764,0.741,0.494
7,XGBoost,P9a,CV_group,0.740,0.700,0.851,0.756,0.713,0.419
8,XGBoost,P9a,independent_test,0.770,0.750,0.800,0.774,0.858,0.543


10.4: SVM

In [ ]:
from sklearn.svm import SVC
eval_sklearn('SVM', lambda: SVC(
    C=1.0, kernel='rbf', probability=True,
    class_weight='balanced', random_state=SEED))

model strain             eval  Accuracy  Precision  Recall    F1   AUC    MCC
  SVM     C4        CV_random     0.640      0.632   0.624 0.625 0.708  0.283
  SVM     C4         CV_group     0.533      0.531   0.704 0.561 0.584  0.148
  SVM     C4 independent_test     0.705      0.667   0.800 0.727 0.740  0.420
  SVM     C5        CV_random     0.782      0.420   0.156 0.210 0.753  0.139
  SVM     C5         CV_group     0.811      0.000   0.000 0.000 0.705 -0.025
  SVM     C5 independent_test     0.803      0.333   0.091 0.143 0.784  0.091
  SVM    P9a        CV_random     0.727      0.704   0.775 0.736 0.762  0.460
  SVM    P9a         CV_group     0.698      0.655   0.782 0.692 0.741  0.336
  SVM    P9a independent_test     0.705      0.676   0.767 0.719 0.783  0.415


,model,strain,eval,Accuracy,Precision,Recall,F1,AUC,MCC
0,SVM,C4,CV_random,0.640,0.632,0.624,0.625,0.708,0.283
1,SVM,C4,CV_group,0.533,0.531,0.704,0.561,0.584,0.148
2,SVM,C4,independent_test,0.705,0.667,0.800,0.727,0.740,0.420
3,SVM,C5,CV_random,0.782,0.420,0.156,0.210,0.753,0.139
4,SVM,C5,CV_group,0.811,0.000,0.000,0.000,0.705,-0.025
5,SVM,C5,independent_test,0.803,0.333,0.091,0.143,0.784,0.091
6,SVM,P9a,CV_random,0.727,0.704,0.775,0.736,0.762,0.460
7,SVM,P9a,CV_group,0.698,0.655,0.782,0.692,0.741,0.336
8,SVM,P9a,independent_test,0.705,0.676,0.767,0.719,0.783,0.415


# 11. Deep learning trên Embedding
Tất cả models sẽ chạy 20 lần lặp để tính ổn định và đúng kỹ thuật

11.1: Thiết lập thông số và các biến chạy các model học sâu trên embedding

In [ ]:
# =====================================================================
# DEEP LEARNING EMBEDDING v2 — eval_deep NHẬN THAM SỐ (X, df, g)
# Không dùng biến global X_full -> gọi rõ nhánh nào, không lo đè biến.
# =====================================================================
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef)

dev='cuda' if torch.cuda.is_available() else 'cpu'
DL_REPEATS = 20
DL_EPOCHS  = 80
N_GENES, EMB_DIM, N_EXTRA = 5, 768, 3   # cố định cho embedding (3843)
print(f'Deep EMB v2 | device:{dev} | lặp:{DL_REPEATS}')

def _metrics(yt, prob, thr=0.5):
    yp=(prob>=thr).astype(int)
    d=dict(Accuracy=accuracy_score(yt,yp),Precision=precision_score(yt,yp,zero_division=0),
           Recall=recall_score(yt,yp,zero_division=0),F1=f1_score(yt,yp,zero_division=0),
           MCC=matthews_corrcoef(yt,yp))
    try: d['AUC']=roc_auc_score(yt,prob)
    except: d['AUC']=np.nan
    return d

def _split_emb(x):
    emb=x[:, :N_GENES*EMB_DIM].view(x.size(0), N_GENES, EMB_DIM)
    extra=x[:, N_GENES*EMB_DIM:]
    return emb, extra

def fit_predict_dl(make_model, Xtr, ytr, Xte, epochs, seed):
    torch.manual_seed(seed)
    m=make_model().to(dev)
    Xt=torch.tensor(Xtr,dtype=torch.float32,device=dev); yt=torch.tensor(ytr,dtype=torch.float32,device=dev)
    Xe=torch.tensor(Xte,dtype=torch.float32,device=dev)
    npos=max(ytr.sum(),1); pw=torch.tensor([(len(ytr)-npos)/npos],dtype=torch.float32,device=dev)
    opt=torch.optim.Adam(m.parameters(),lr=1e-3,weight_decay=1e-3); lf=nn.BCEWithLogitsLoss(pos_weight=pw)
    m.train()
    for _ in range(epochs):
        opt.zero_grad(); lf(m(Xt).squeeze(-1),yt).backward(); opt.step()
    m.eval()
    with torch.no_grad(): p=torch.sigmoid(m(Xe).squeeze(-1)).cpu().numpy()
    del m,Xt,yt,Xe; torch.cuda.empty_cache()
    return p

# === eval_deep NHẬN THAM SỐ X, df, g (không dùng global) ===
def eval_deep(name, make_model, X, df, g, out_dir, epochs=DL_EPOCHS, repeats=DL_REPEATS, seed=42):
    BIN=[s for s in ['C4','C5','P9a'] if s+'_bin' in df.columns]
    rows=[]; t0=time.time()
    for s in BIN:
        y=df[s+'_bin'].values.astype(int); idx=np.arange(len(y))
        tr,te=train_test_split(idx,test_size=0.20,stratify=y,random_state=seed)
        Xtr,Xte=X[tr],X[te]; ytr,yte=y[tr],y[te]; gtr=g[tr]
        for cvname,cv,grp in [('CV_random',StratifiedKFold(5,shuffle=True,random_state=seed),None),
                              ('CV_group',GroupKFold(5),gtr)]:
            rep_m=[]
            for rep in range(repeats):
                fold=[]
                for a,b in cv.split(Xtr,ytr,grp):
                    if len(np.unique(ytr[b]))<2: continue
                    sc=StandardScaler().fit(Xtr[a])
                    prob=fit_predict_dl(make_model,sc.transform(Xtr[a]),ytr[a],sc.transform(Xtr[b]),epochs,rep)
                    fold.append(_metrics(ytr[b],prob))
                rep_m.append(pd.DataFrame(fold).mean())
            dm=pd.DataFrame(rep_m); r={'model':name,'strain':s,'eval':cvname}
            for k in ['Accuracy','Precision','Recall','F1','AUC','MCC']:
                r[k]=round(dm[k].mean(),3); r[k+'_std']=round(dm[k].std(),3)
            rows.append(r)
        tst=[]
        for rep in range(repeats):
            sc=StandardScaler().fit(Xtr)
            prob=fit_predict_dl(make_model,sc.transform(Xtr),ytr,sc.transform(Xte),epochs,rep)
            tst.append(_metrics(yte,prob))
        dm=pd.DataFrame(tst); r={'model':name,'strain':s,'eval':'independent_test'}
        for k in ['Accuracy','Precision','Recall','F1','AUC','MCC']:
            r[k]=round(dm[k].mean(),3); r[k+'_std']=round(dm[k].std(),3)
        rows.append(r)
    df_res=pd.DataFrame(rows); df_res.to_csv(rf'{out_dir}\train_{name}.csv',index=False)
    print(df_res[['model','strain','eval','Accuracy','F1','AUC','MCC']].to_string(index=False))
    print(f'\n{name} xong: {(time.time()-t0)/60:.1f} phút'); return df_res

# ---------- KIẾN TRÚC embedding ----------
class MLP_emb(nn.Module):
    def __init__(s):
        super().__init__()
        s.net=nn.Sequential(nn.Linear(N_GENES*EMB_DIM+N_EXTRA,256),nn.ReLU(),nn.BatchNorm1d(256),nn.Dropout(0.4),
                            nn.Linear(256,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x): return s.net(x)
class CNN1D_emb(nn.Module):
    def __init__(s):
        super().__init__()
        s.conv=nn.Sequential(nn.Conv1d(N_GENES,32,7,padding=3),nn.ReLU(),nn.MaxPool1d(4),
            nn.Conv1d(32,64,5,padding=2),nn.ReLU(),nn.AdaptiveAvgPool1d(8))
        s.head=nn.Sequential(nn.Linear(64*8+N_EXTRA,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x):
        emb,extra=_split_emb(x); return s.head(torch.cat([s.conv(emb).flatten(1),extra],1))
class CNN2D_emb(nn.Module):
    def __init__(s):
        super().__init__()
        s.conv=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d((1,4)),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d((5,8)))
        s.head=nn.Sequential(nn.Linear(32*5*8+N_EXTRA,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x):
        emb,extra=_split_emb(x); return s.head(torch.cat([s.conv(emb.unsqueeze(1)).flatten(1),extra],1))
class mCNN_emb(nn.Module):
    def __init__(s,kernels=(3,7,15),nfilt=32):
        super().__init__()
        s.br=nn.ModuleList([nn.Conv1d(N_GENES,nfilt,k,padding=k//2) for k in kernels])
        s.head=nn.Sequential(nn.Linear(nfilt*len(kernels)+N_EXTRA,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x):
        emb,extra=_split_emb(x); feats=[torch.relu(b(emb)).max(dim=2).values for b in s.br]
        return s.head(torch.cat(feats+[extra],1))

print('Deep EMB v2 OK: eval_deep(name,model,X,df,g,out_dir) + MLP_emb,CNN1D_emb,CNN2D_emb,mCNN_emb')

Deep EMB v2 | device:cuda | lặp:20
Deep EMB v2 OK: eval_deep(name,model,X,df,g,out_dir) + MLP_emb,CNN1D_emb,CNN2D_emb,mCNN_emb


Kiểm tra các thông số đã có để chuẩn bị chạy

In [ ]:
import numpy as np, pandas as pd
OUT = r'C:\GenBBPipeline\Export'; SEED = 42
z = np.load(rf'{OUT}\X_full_14locus.npz', allow_pickle=True)
X_emb = z['X_full']                    # tên riêng X_emb
df_emb = pd.read_pickle(rf'{OUT}\df_keep_14locus.pkl')
g_emb = df_emb['subgroup'].values
print('X_emb:', X_emb.shape)           # (303, 3843)

X_emb: (303, 3843)


11.2: MLP

In [ ]:
# =====================================================================
# CELL MLP — model deep trên embedding. LẶP 20 LẦN (mean±std).
# Weighted loss (pos_weight) cho công bằng với balanced của model ML.
# =====================================================================
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import StandardScaler

dev='cuda' if torch.cuda.is_available() else 'cpu'
MLP_REPEATS = 20
print('MLP device:', dev, '| lặp:', MLP_REPEATS)

def make_mlp(n_in):
    return nn.Sequential(nn.Linear(n_in,256),nn.ReLU(),nn.BatchNorm1d(256),nn.Dropout(0.4),
                         nn.Linear(256,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))

def mlp_fit_predict(Xtr, ytr, Xte, seed):
    torch.manual_seed(seed)
    m=make_mlp(Xtr.shape[1]).to(dev)
    Xt=torch.tensor(Xtr,dtype=torch.float32,device=dev); yt=torch.tensor(ytr,dtype=torch.float32,device=dev)
    Xe=torch.tensor(Xte,dtype=torch.float32,device=dev)
    # pos_weight = n_am / n_duong (tuong duong class_weight balanced)
    npos=max(ytr.sum(),1); nneg=max(len(ytr)-ytr.sum(),1)
    pw=torch.tensor([nneg/npos],dtype=torch.float32,device=dev)
    opt=torch.optim.Adam(m.parameters(),lr=1e-3,weight_decay=1e-3)
    lf=nn.BCEWithLogitsLoss(pos_weight=pw)
    m.train()
    for _ in range(80):
        opt.zero_grad(); lf(m(Xt).squeeze(-1),yt).backward(); opt.step()
    m.eval()
    with torch.no_grad(): return torch.sigmoid(m(Xe).squeeze(-1)).cpu().numpy()

def eval_mlp():
    rows=[]
    for s in BIN:
        tr,te,y=get_split(s)
        Xtr,Xte=X_full[tr],X_full[te]; ytr,yte=y[tr],y[te]; gtr=g_full[tr]
        for cvname,cv,grp in [('CV_random',StratifiedKFold(5,shuffle=True,random_state=SEED),None),
                              ('CV_group',GroupKFold(5),gtr)]:
            # lặp MLP_REPEATS lần, mỗi lần seed khác -> lấy mean
            rep_metrics=[]
            for rep in range(MLP_REPEATS):
                fold_acc=[]
                for a,b in cv.split(Xtr,ytr,grp):
                    if len(np.unique(ytr[b]))<2: continue
                    sc=StandardScaler().fit(Xtr[a])
                    prob=mlp_fit_predict(sc.transform(Xtr[a]),ytr[a],sc.transform(Xtr[b]),seed=rep)
                    fold_acc.append(_metrics(ytr[b],prob))
                rep_metrics.append(pd.DataFrame(fold_acc).mean())
            dm=pd.DataFrame(rep_metrics)
            r={'model':'MLP','strain':s,'eval':cvname}
            for k in ['Accuracy','Precision','Recall','F1','AUC','MCC']:
                r[k]=round(dm[k].mean(),3); r[k+'_std']=round(dm[k].std(),3)
            rows.append(r)
        # test độc lập: lặp 20 lần rồi mean
        tst=[]
        for rep in range(MLP_REPEATS):
            sc=StandardScaler().fit(Xtr)
            prob=mlp_fit_predict(sc.transform(Xtr),ytr,sc.transform(Xte),seed=rep)
            tst.append(_metrics(yte,prob))
        dm=pd.DataFrame(tst)
        r={'model':'MLP','strain':s,'eval':'independent_test'}
        for k in ['Accuracy','Precision','Recall','F1','AUC','MCC']:
            r[k]=round(dm[k].mean(),3); r[k+'_std']=round(dm[k].std(),3)
        rows.append(r)
    df=pd.DataFrame(rows); df.to_csv(f'{OUT}/train_MLP.csv',index=False)
    # in gọn (chỉ mean các chỉ số chính)
    cols=['model','strain','eval','Accuracy','F1','AUC','MCC']
    print(df[cols].to_string(index=False))
    print('\n(std lưu trong file train_MLP.csv)')
    return df

t0=time.time()
df_mlp=eval_mlp()
print(f'\nMLP xong trong {(time.time()-t0)/60:.1f} phút')

MLP device: cuda | lặp: 20
model strain             eval  Accuracy    F1   AUC   MCC
  MLP     C4        CV_random     0.658 0.643 0.729 0.317
  MLP     C4         CV_group     0.592 0.568 0.611 0.168
  MLP     C4 independent_test     0.605 0.623 0.648 0.213
  MLP     C5        CV_random     0.749 0.357 0.684 0.206
  MLP     C5         CV_group     0.735 0.260 0.609 0.078
  MLP     C5 independent_test     0.770 0.432 0.710 0.293
  MLP    P9a        CV_random     0.690 0.688 0.731 0.379
  MLP    P9a         CV_group     0.694 0.686 0.689 0.309
  MLP    P9a independent_test     0.652 0.629 0.766 0.305

(std lưu trong file train_MLP.csv)

MLP xong trong 2.2 phút


11.3: CNN-1D trên embedding

In [ ]:
eval_deep('CNN1D_emb', CNN1D_emb, X_emb, df_emb, g_emb, OUT)

    model strain             eval  Accuracy    F1   AUC   MCC
CNN1D_emb     C4        CV_random     0.608 0.577 0.661 0.217
CNN1D_emb     C4         CV_group     0.524 0.526 0.569 0.081
CNN1D_emb     C4 independent_test     0.639 0.644 0.694 0.281
CNN1D_emb     C5        CV_random     0.681 0.362 0.683 0.179
CNN1D_emb     C5         CV_group     0.649 0.320 0.599 0.102
CNN1D_emb     C5 independent_test     0.627 0.378 0.671 0.198
CNN1D_emb    P9a        CV_random     0.708 0.709 0.732 0.424
CNN1D_emb    P9a         CV_group     0.699 0.694 0.689 0.325
CNN1D_emb    P9a independent_test     0.657 0.650 0.669 0.315

CNN1D_emb xong: 2.5 phút


,model,strain,eval,Accuracy,Accuracy_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,AUC,AUC_std,MCC,MCC_std
0,CNN1D_emb,C4,CV_random,0.608,0.021,0.606,0.024,0.564,0.054,0.577,0.035,0.661,0.016,0.217,0.042
1,CNN1D_emb,C4,CV_group,0.524,0.023,0.513,0.016,0.613,0.038,0.526,0.021,0.569,0.018,0.081,0.038
2,CNN1D_emb,C4,independent_test,0.639,0.046,0.629,0.046,0.665,0.067,0.644,0.042,0.694,0.025,0.281,0.091
3,CNN1D_emb,C5,CV_random,0.681,0.011,0.308,0.020,0.468,0.069,0.362,0.032,0.683,0.018,0.179,0.040
4,CNN1D_emb,C5,CV_group,0.649,0.018,0.252,0.027,0.476,0.049,0.320,0.031,0.599,0.015,0.102,0.036
5,CNN1D_emb,C5,independent_test,0.627,0.050,0.273,0.033,0.623,0.044,0.378,0.030,0.671,0.021,0.198,0.048
6,CNN1D_emb,P9a,CV_random,0.708,0.012,0.691,0.011,0.736,0.025,0.709,0.015,0.732,0.008,0.424,0.025
7,CNN1D_emb,P9a,CV_group,0.699,0.011,0.687,0.021,0.724,0.034,0.694,0.018,0.689,0.013,0.325,0.028
8,CNN1D_emb,P9a,independent_test,0.657,0.030,0.655,0.042,0.650,0.063,0.650,0.031,0.669,0.023,0.315,0.059


11.4: CNN-2D trên embedding

In [ ]:
eval_deep('CNN2D_emb', CNN2D_emb, X_emb, df_emb, g_emb, OUT)

    model strain             eval  Accuracy    F1   AUC   MCC
CNN2D_emb     C4        CV_random     0.630 0.602 0.678 0.262
CNN2D_emb     C4         CV_group     0.534 0.533 0.560 0.082
CNN2D_emb     C4 independent_test     0.666 0.678 0.722 0.337
CNN2D_emb     C5        CV_random     0.688 0.413 0.717 0.244
CNN2D_emb     C5         CV_group     0.642 0.361 0.635 0.149
CNN2D_emb     C5 independent_test     0.557 0.342 0.695 0.136
CNN2D_emb    P9a        CV_random     0.700 0.705 0.726 0.410
CNN2D_emb    P9a         CV_group     0.704 0.692 0.680 0.323
CNN2D_emb    P9a independent_test     0.670 0.680 0.707 0.344

CNN2D_emb xong: 5.6 phút


,model,strain,eval,Accuracy,Accuracy_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,AUC,AUC_std,MCC,MCC_std
0,CNN2D_emb,C4,CV_random,0.630,0.019,0.629,0.025,0.586,0.031,0.602,0.021,0.678,0.014,0.262,0.039
1,CNN2D_emb,C4,CV_group,0.534,0.020,0.517,0.019,0.618,0.057,0.533,0.028,0.560,0.024,0.082,0.047
2,CNN2D_emb,C4,independent_test,0.666,0.037,0.643,0.034,0.722,0.077,0.678,0.043,0.722,0.033,0.337,0.076
3,CNN2D_emb,C5,CV_random,0.688,0.018,0.336,0.017,0.566,0.044,0.413,0.018,0.717,0.010,0.244,0.027
4,CNN2D_emb,C5,CV_group,0.642,0.014,0.275,0.028,0.571,0.054,0.361,0.030,0.635,0.014,0.149,0.035
5,CNN2D_emb,C5,independent_test,0.557,0.036,0.235,0.019,0.636,0.000,0.342,0.020,0.695,0.014,0.136,0.035
6,CNN2D_emb,P9a,CV_random,0.700,0.010,0.682,0.011,0.740,0.027,0.705,0.014,0.726,0.006,0.410,0.021
7,CNN2D_emb,P9a,CV_group,0.704,0.012,0.692,0.027,0.715,0.040,0.692,0.026,0.680,0.011,0.323,0.039
8,CNN2D_emb,P9a,independent_test,0.670,0.031,0.652,0.031,0.712,0.050,0.680,0.031,0.707,0.021,0.344,0.062


11.5: mCNN trên embedding

In [ ]:
eval_deep('mCNN_emb', mCNN_emb, X_emb, df_emb, g_emb, OUT)

   model strain             eval  Accuracy    F1   AUC    MCC
mCNN_emb     C4        CV_random     0.612 0.579 0.665  0.222
mCNN_emb     C4         CV_group     0.500 0.521 0.545  0.066
mCNN_emb     C4 independent_test     0.652 0.679 0.736  0.314
mCNN_emb     C5        CV_random     0.756 0.260 0.606  0.125
mCNN_emb     C5         CV_group     0.718 0.133 0.491 -0.050
mCNN_emb     C5 independent_test     0.777 0.335 0.642  0.210
mCNN_emb    P9a        CV_random     0.641 0.651 0.694  0.285
mCNN_emb    P9a         CV_group     0.603 0.600 0.621  0.154
mCNN_emb    P9a independent_test     0.575 0.581 0.575  0.153

mCNN_emb xong: 2.7 phút


,model,strain,eval,Accuracy,Accuracy_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,AUC,AUC_std,MCC,MCC_std
0,mCNN_emb,C4,CV_random,0.612,0.021,0.608,0.023,0.559,0.040,0.579,0.028,0.665,0.011,0.222,0.041
1,mCNN_emb,C4,CV_group,0.500,0.029,0.495,0.023,0.643,0.041,0.521,0.028,0.545,0.018,0.066,0.051
2,mCNN_emb,C4,independent_test,0.652,0.037,0.623,0.035,0.748,0.038,0.679,0.032,0.736,0.025,0.314,0.074
3,mCNN_emb,C5,CV_random,0.756,0.014,0.308,0.051,0.236,0.055,0.260,0.052,0.606,0.019,0.125,0.055
4,mCNN_emb,C5,CV_group,0.718,0.020,0.155,0.065,0.124,0.047,0.133,0.046,0.491,0.021,-0.050,0.044
5,mCNN_emb,C5,independent_test,0.777,0.042,0.374,0.141,0.318,0.120,0.335,0.112,0.642,0.045,0.210,0.131
6,mCNN_emb,P9a,CV_random,0.641,0.021,0.630,0.023,0.679,0.029,0.651,0.020,0.694,0.013,0.285,0.043
7,mCNN_emb,P9a,CV_group,0.603,0.023,0.582,0.030,0.648,0.045,0.600,0.021,0.621,0.021,0.154,0.043
8,mCNN_emb,P9a,independent_test,0.575,0.034,0.565,0.033,0.600,0.061,0.581,0.038,0.575,0.035,0.153,0.069


# ĐẶC BIỆT: Lưu thư mục cho model

1. Lưu ML + MLP trên embedding

In [ ]:
# =====================================================================
# LƯU 5 MODEL (14 locus, X_full) cho teammate import vào web workflow
# sklearn -> .joblib (model + scaler)   |   MLP -> .pt (state_dict + arch)
# Train trên TOÀN BỘ dữ liệu (X_full) cho mỗi chủng.
# =====================================================================
import os, json, joblib, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

MODEL_DIR = f'{OUT}/models_14locus'
os.makedirs(MODEL_DIR, exist_ok=True)
dev='cuda' if torch.cuda.is_available() else 'cpu'
manifest=[]
BINS=[s for s in ['C4','C5','P9a'] if s+'_bin' in df_keep.columns]

def save_sk(name, make_model, strain):
    y=df_keep[strain+'_bin'].values.astype(int)
    sc=StandardScaler().fit(X_full)
    m=make_model(); m.fit(sc.transform(X_full), y)
    fn=f'{MODEL_DIR}/{name}_{strain}.joblib'
    joblib.dump({'model':m,'scaler':sc,'strain':strain,'n_features':X_full.shape[1],
                 'feature':'X_full_14locus','model_type':name}, fn)
    manifest.append(dict(file=os.path.basename(fn),model=name,strain=strain,framework='sklearn'))
    print(f'  {os.path.basename(fn)}')

# --- MLP ---
def make_mlp(n_in):
    return nn.Sequential(nn.Linear(n_in,256),nn.ReLU(),nn.BatchNorm1d(256),nn.Dropout(0.4),
                         nn.Linear(256,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
def save_mlp(strain):
    y=df_keep[strain+'_bin'].values.astype(int)
    sc=StandardScaler().fit(X_full)
    Xt=torch.tensor(sc.transform(X_full),dtype=torch.float32,device=dev)
    yt=torch.tensor(y,dtype=torch.float32,device=dev)
    npos=max(y.sum(),1); pw=torch.tensor([(len(y)-npos)/npos],dtype=torch.float32,device=dev)
    m=make_mlp(X_full.shape[1]).to(dev); m.train()
    opt=torch.optim.Adam(m.parameters(),lr=1e-3,weight_decay=1e-3); lf=nn.BCEWithLogitsLoss(pos_weight=pw)
    for _ in range(80): opt.zero_grad(); lf(m(Xt).squeeze(-1),yt).backward(); opt.step()
    torch.save({'state_dict':m.state_dict(),'arch':{'n_in':X_full.shape[1],'hidden':[256,64]},
                'strain':strain,'model_type':'MLP'}, f'{MODEL_DIR}/MLP_{strain}.pt')
    joblib.dump(sc, f'{MODEL_DIR}/MLP_{strain}_scaler.joblib')
    manifest.append(dict(file=f'MLP_{strain}.pt',model='MLP',strain=strain,framework='torch'))
    print(f'  MLP_{strain}.pt')

SK={'LogReg':lambda:LogisticRegression(C=0.05,max_iter=3000,class_weight='balanced',random_state=SEED),
    'RandomForest':lambda:RandomForestClassifier(n_estimators=300,max_depth=15,max_features='sqrt',
        min_samples_leaf=5,class_weight='balanced',n_jobs=-1,random_state=SEED),
    'SVM':lambda:SVC(C=1.0,kernel='rbf',probability=True,class_weight='balanced',random_state=SEED)}
try:
    from xgboost import XGBClassifier
    spw=np.mean([(df_keep[s+'_bin']==0).sum()/max((df_keep[s+'_bin']==1).sum(),1) for s in BINS])
    SK['XGBoost']=lambda:XGBClassifier(n_estimators=300,max_depth=4,learning_rate=0.05,subsample=0.8,
        colsample_bytree=0.5,scale_pos_weight=spw,n_jobs=-1,random_state=SEED,verbosity=0,eval_metric='logloss')
except: pass

print('=== Lưu model ===')
for s in BINS:
    for name,mk in SK.items(): save_sk(name,mk,s)
    save_mlp(s)

# --- README (quan trọng: mô tả định dạng đầu vào cho teammate) ---
readme=f"""# MODELS 14-LOCUS — Dự đoán kháng bạc lá lúa

## Định dạng ĐẦU VÀO (bắt buộc đọc)
Model nhận vector đặc trưng X_full, {X_full.shape[1]} chiều, theo thứ tự:
  [1] embedding 5 gene (Plant-DNABERT, 768 chiều/gene, mean-pool):
      xa5, xa13_SWEET11, xa25_SWEET13, Xa21, Xa1  -> 5 x 768 = 3840 chiều
  [2] Xa1_present: 1 cột (1 nếu Xa1 sạch, 0 nếu imputation)
  [3] presence/absence: Xa23, Xa48 -> 2 cột (0/1)
  Tổng: {X_full.shape[1]} chiều. PHẢI chuẩn hóa bằng scaler kèm theo.

## Tên file: {{model}}_{{strain}}.{{ext}}   (strain = C4/C5/P9a)

## Load sklearn:
    import joblib
    d=joblib.load('XGBoost_P9a.joblib')
    prob=d['model'].predict_proba(d['scaler'].transform(X_new))[:,1]

## Load MLP (torch):
    import torch, joblib
    ck=torch.load('MLP_P9a.pt')
    # tái tạo kiến trúc theo ck['arch'], load_state_dict(ck['state_dict'])
    sc=joblib.load('MLP_P9a_scaler.joblib')

## LƯU Ý
Model minh họa pipeline (xuất & mở rộng được). Hiệu năng: P9a tốt nhất
(AUC~0.83), C5 khó nhất (mất cân bằng). Báo cáo MCC/AUC, không chỉ accuracy.
Tổng model: {len(manifest)}
"""
open(f'{MODEL_DIR}/README.txt','w').write(readme)
pd.DataFrame(manifest).to_csv(f'{MODEL_DIR}/manifest.csv',index=False)

import shutil
shutil.make_archive(f'{OUT}/models_14locus', 'zip', MODEL_DIR)
print(f'\n=== XONG. {len(manifest)} model ===')
print(f'>>> File gửi teammate: {OUT}/models_14locus.zip <<<')
print(f'    (chứa .joblib, .pt, scaler, README, manifest)')

=== Lưu model ===
  LogReg_C4.joblib
  RandomForest_C4.joblib
  SVM_C4.joblib
  XGBoost_C4.joblib
  MLP_C4.pt
  LogReg_C5.joblib
  RandomForest_C5.joblib
  SVM_C5.joblib
  XGBoost_C5.joblib
  MLP_C5.pt
  LogReg_P9a.joblib
  RandomForest_P9a.joblib
  SVM_P9a.joblib
  XGBoost_P9a.joblib
  MLP_P9a.pt

=== XONG. 15 model ===
>>> File gửi teammate: /content/drive/MyDrive/Project/Bioinformatics - Thesis/Export/models_14locus.zip <<<
    (chứa .joblib, .pt, scaler, README, manifest)


2. Lưu các model CNN và mCNN trên embedding

In [45]:
# =====================================================================
# LƯU MODEL DEEP EMBEDDING (CNN1D/2D, mCNN, MLP) cho teammate
# Chạy trên máy thuê, cần X_emb (embedding 3843) + df_emb trong RAM.
# =====================================================================
import os, joblib, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler

OUT = r'C:\GenBBPipeline\Export'
MODEL_DIR = rf'{OUT}\models_emb_deep'
os.makedirs(MODEL_DIR, exist_ok=True)
dev='cuda' if torch.cuda.is_available() else 'cpu'
SEED=42
N_GENES, EMB_DIM, N_EXTRA = 5, 768, 3
BINS=[s for s in ['C4','C5','P9a'] if s+'_bin' in df_emb.columns]
manifest=[]

def _split_emb(x):
    emb=x[:, :N_GENES*EMB_DIM].view(x.size(0),N_GENES,EMB_DIM); extra=x[:, N_GENES*EMB_DIM:]
    return emb, extra

# ---- kiến trúc (khớp emb_deep_v2) ----
class MLP_emb(nn.Module):
    def __init__(s):
        super().__init__()
        s.net=nn.Sequential(nn.Linear(N_GENES*EMB_DIM+N_EXTRA,256),nn.ReLU(),nn.BatchNorm1d(256),nn.Dropout(0.4),
                            nn.Linear(256,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x): return s.net(x)
class CNN1D_emb(nn.Module):
    def __init__(s):
        super().__init__()
        s.conv=nn.Sequential(nn.Conv1d(N_GENES,32,7,padding=3),nn.ReLU(),nn.MaxPool1d(4),
            nn.Conv1d(32,64,5,padding=2),nn.ReLU(),nn.AdaptiveAvgPool1d(8))
        s.head=nn.Sequential(nn.Linear(64*8+N_EXTRA,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x):
        emb,extra=_split_emb(x); return s.head(torch.cat([s.conv(emb).flatten(1),extra],1))
class CNN2D_emb(nn.Module):
    def __init__(s):
        super().__init__()
        s.conv=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d((1,4)),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d((5,8)))
        s.head=nn.Sequential(nn.Linear(32*5*8+N_EXTRA,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x):
        emb,extra=_split_emb(x); return s.head(torch.cat([s.conv(emb.unsqueeze(1)).flatten(1),extra],1))
class mCNN_emb(nn.Module):
    def __init__(s,kernels=(3,7,15),nfilt=32):
        super().__init__()
        s.br=nn.ModuleList([nn.Conv1d(N_GENES,nfilt,k,padding=k//2) for k in kernels])
        s.head=nn.Sequential(nn.Linear(nfilt*len(kernels)+N_EXTRA,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x):
        emb,extra=_split_emb(x); feats=[torch.relu(b(emb)).max(dim=2).values for b in s.br]
        return s.head(torch.cat(feats+[extra],1))

DEEP={'MLP_emb':MLP_emb,'CNN1D_emb':CNN1D_emb,'CNN2D_emb':CNN2D_emb,'mCNN_emb':mCNN_emb}
print('=== Lưu DEEP embedding models (train full + save) ===')
for s in BINS:
    y=df_emb[s+'_bin'].values.astype(int)
    sc=StandardScaler().fit(X_emb)
    Xt=torch.tensor(sc.transform(X_emb),dtype=torch.float32,device=dev)
    yt=torch.tensor(y,dtype=torch.float32,device=dev)
    npos=max(y.sum(),1); pw=torch.tensor([(len(y)-npos)/npos],dtype=torch.float32,device=dev)
    for name,cls in DEEP.items():
        torch.manual_seed(SEED); m=cls().to(dev); m.train()
        opt=torch.optim.Adam(m.parameters(),lr=1e-3,weight_decay=1e-3); lf=nn.BCEWithLogitsLoss(pos_weight=pw)
        for _ in range(80): opt.zero_grad(); lf(m(Xt).squeeze(-1),yt).backward(); opt.step()
        torch.save({'state_dict':m.state_dict(),'arch':name,'n_in':X_emb.shape[1],
                    'n_genes':N_GENES,'emb_dim':EMB_DIM,'n_extra':N_EXTRA,'strain':s},
                   rf'{MODEL_DIR}\{name}_{s}.pt')
        joblib.dump(sc, rf'{MODEL_DIR}\{name}_{s}_scaler.joblib')
        manifest.append(dict(file=f'{name}_{s}.pt',model=name,strain=s,framework='torch'))
    print(f'  {s}: {list(DEEP.keys())}')

pd.DataFrame(manifest).to_csv(rf'{MODEL_DIR}\manifest.csv',index=False)
readme=f"""# MODELS DEEP EMBEDDING (CNN1D/2D, mCNN, MLP)

## Input: X_full 3843 chiều (giống XGBoost embedding — xem BioFlow spec)
   = 5 gene×768 embedding + Xa1_present + Xa23 + Xa48
   Reshape trong model: 3840 -> (5 gene, 768), 3 extra nối ở FC.

## File: {{model}}_{{strain}}.pt (state_dict + arch) + _scaler.joblib
   arch info trong .pt: n_genes=5, emb_dim=768, n_extra=3

## LOAD
   ck=torch.load('CNN1D_emb_P9a.pt')
   # tái tạo class CNN1D_emb theo arch (định nghĩa trong notebook)
   # m.load_state_dict(ck['state_dict']); sc=joblib.load('CNN1D_emb_P9a_scaler.joblib')
   # prob = sigmoid(m(scaler.transform(X)))

## LƯU Ý: kết quả deep < classical (RF/XGBoost). Xem results_ALL_2branch.csv.
   Model minh họa pipeline, không phải predictor tốt nhất.
"""
open(rf'{MODEL_DIR}\README_emb_deep.txt','w',encoding='utf-8').write(readme)

import shutil
shutil.make_archive(rf'{OUT}\models_emb_deep','zip',MODEL_DIR)
print(f'\n=== XONG. {len(manifest)} model deep embedding ===')
print(f'>>> Gửi teammate: {OUT}\\models_emb_deep.zip <<<')
print('⚠️ TẢI VỀ trước khi trả máy!')

=== Lưu DEEP embedding models (train full + save) ===
  C4: ['MLP_emb', 'CNN1D_emb', 'CNN2D_emb', 'mCNN_emb']
  C5: ['MLP_emb', 'CNN1D_emb', 'CNN2D_emb', 'mCNN_emb']
  P9a: ['MLP_emb', 'CNN1D_emb', 'CNN2D_emb', 'mCNN_emb']

=== XONG. 12 model deep embedding ===
>>> Gửi teammate: C:\GenBBPipeline\Export\models_emb_deep.zip <<<
⚠️ TẢI VỀ trước khi trả máy!


## 12. Thiết lập các thông số và nạp SNP
Đây là phần chạy để nạp SNP vào để train mô hình

In [ ]:
# =====================================================================
# NHÁNH SNP — CÁCH 2: nạp genotype + lọc 327 mẫu + MAF>0.05 + 100k top-variance
# Xử lý theo KHỐI SNP để không tràn RAM (4.82M SNP × 327 mẫu)
# =====================================================================
import numpy as np, pandas as pd, time
from bed_reader import open_bed

# ---- đường dẫn ----
GENO_PREFIX = r'C:\GenBBPipeline\Bacterial Leaf Data\genotype\base_filtered_v0.7'
OUT         = r'C:\GenBBPipeline\Export'
MAF_MIN     = 0.05
N_KEEP_SNP  = 100_000
CHUNK       = 200_000        # số SNP đọc mỗi khối (điều chỉnh nếu RAM căng)

# ---- nạp df_keep (327 mẫu + nhãn) ----
df_keep = pd.read_pickle(rf'{OUT}\df_keep_14locus.pkl')
keep_ids = list(df_keep['id'].astype(str))
print(f'df_keep: {len(keep_ids)} mẫu')

# ---- mở genotype, tìm index của 327 mẫu ----
bed = open_bed(GENO_PREFIX + '.bed')
all_iid = list(bed.iid)
id_to_idx = {s:i for i,s in enumerate(all_iid)}
miss = [s for s in keep_ids if s not in id_to_idx]
if miss: print(f'⚠️ {len(miss)} mẫu không có trong genotype: {miss[:5]}...')
iid_idx = np.array([id_to_idx[s] for s in keep_ids if s in id_to_idx])
keep_ids_ok = [s for s in keep_ids if s in id_to_idx]
print(f'Khớp {len(iid_idx)}/{len(keep_ids)} mẫu trong genotype')

n_snp = bed.sid_count
print(f'Tổng SNP: {n_snp:,} | đọc theo khối {CHUNK:,}')

# =====================================================================
# PASS 1: quét theo khối, tính MAF + variance mỗi SNP (chỉ 327 mẫu)
# =====================================================================
maf = np.zeros(n_snp, dtype='float32')
var = np.zeros(n_snp, dtype='float32')
t0=time.time()
for start in range(0, n_snp, CHUNK):
    end = min(start+CHUNK, n_snp)
    # đọc [327 mẫu × khối SNP], giá trị 0/1/2, NaN nếu thiếu
    G = bed.read(index=np.s_[iid_idx, start:end], dtype='float32')
    p = np.nanmean(G, axis=0) / 2.0          # tần số allele
    maf[start:end] = np.minimum(p, 1-p)      # MAF
    var[start:end] = np.nanvar(G, axis=0)    # phương sai
    del G
    if (start//CHUNK) % 5 == 0:
        el=time.time()-t0
        print(f'  {end:,}/{n_snp:,} | {el:.0f}s', flush=True)
print(f'Pass 1 xong: {time.time()-t0:.0f}s')

# =====================================================================
# CHỌN SNP: MAF > 0.05, rồi top-variance
# =====================================================================
pass_maf = np.where(maf > MAF_MIN)[0]
print(f'SNP qua MAF>{MAF_MIN}: {len(pass_maf):,}')
# trong nhóm qua MAF, chọn top variance
order = pass_maf[np.argsort(var[pass_maf])[::-1]]
sel = np.sort(order[:N_KEEP_SNP])            # giữ thứ tự genome
print(f'Chọn {len(sel):,} SNP (top-variance sau MAF)')

# =====================================================================
# PASS 2: đọc CHỈ các SNP đã chọn cho 327 mẫu -> ma trận cuối
# =====================================================================
X_snp = bed.read(index=np.s_[iid_idx, sel], dtype='float32')
# điền NaN (thiếu) bằng trung bình cột
col_mean = np.nanmean(X_snp, axis=0)
inds = np.where(np.isnan(X_snp))
X_snp[inds] = np.take(col_mean, inds[1])
print(f'X_snp: {X_snp.shape} (327 mẫu × {len(sel)} SNP)')

# ---- căn nhãn theo keep_ids_ok (đúng thứ tự mẫu đã đọc) ----
df_snp_keep = df_keep.set_index('id').loc[keep_ids_ok].reset_index()
g_snp = df_snp_keep['subgroup'].values

# ---- lưu ----
np.savez_compressed(rf'{OUT}\X_snp_genome.npz',
                    X_snp=X_snp, snp_idx=sel, ids=np.array(keep_ids_ok))
df_snp_keep.to_pickle(rf'{OUT}\df_snp_keep.pkl')
print(f'\nĐã lưu X_snp_genome.npz + df_snp_keep.pkl')
print('Khớp:', X_snp.shape[0]==len(df_snp_keep)==len(g_snp))
print('CHECKPOINT SNP-load:', 'PASS' if X_snp.shape[0]>=300 and X_snp.shape[1]==len(sel) else 'FAIL')

df_keep: 303 mẫu
Khớp 303/303 mẫu trong genotype
Tổng SNP: 4,817,964 | đọc theo khối 200,000
  200,000/4,817,964 | 1s
  1,200,000/4,817,964 | 8s
  2,200,000/4,817,964 | 15s
  3,200,000/4,817,964 | 22s
  4,200,000/4,817,964 | 29s
Pass 1 xong: 33s
SNP qua MAF>0.05: 3,315,874
Chọn 100,000 SNP (top-variance sau MAF)
X_snp: (303, 100000) (327 mẫu × 100000 SNP)

Đã lưu X_snp_genome.npz + df_snp_keep.pkl
Khớp: True
CHECKPOINT SNP-load: PASS


Nạp các biến chạy SNP trên máy

In [ ]:
# nạp dữ liệu SNP
import numpy as np, pandas as pd
OUT = r'C:\GenBBPipeline\Export'
z = np.load(rf'{OUT}\X_snp_genome.npz', allow_pickle=True)
X_full = z['X_snp']                                    # dùng tên X_full để khớp cell nền
df_keep = pd.read_pickle(rf'{OUT}\df_snp_keep.pkl')
g_full = df_keep['subgroup'].values
SEED = 42
print('X_snp nạp thành X_full:', X_full.shape)

X_snp nạp thành X_full: (303, 100000)


Định nghĩa các biến

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef)

TEST_SIZE = 0.20
BIN = [s for s in ['C4','C5','P9a'] if s+'_bin' in df_keep.columns]
print(f'X: {X_full.shape} | chủng: {BIN}')

def _metrics(yt, prob, thr=0.5):
    yp=(prob>=thr).astype(int)
    d=dict(Accuracy=accuracy_score(yt,yp),Precision=precision_score(yt,yp,zero_division=0),
           Recall=recall_score(yt,yp,zero_division=0),F1=f1_score(yt,yp,zero_division=0),
           MCC=matthews_corrcoef(yt,yp))
    try: d['AUC']=roc_auc_score(yt,prob)
    except: d['AUC']=np.nan
    return d

def get_split(strain):
    y=df_keep[strain+'_bin'].values.astype(int); idx=np.arange(len(y))
    tr,te=train_test_split(idx,test_size=TEST_SIZE,stratify=y,random_state=SEED)
    return tr,te,y

def eval_sklearn(name, make_model):
    rows=[]
    for s in BIN:
        tr,te,y=get_split(s)
        Xtr,Xte=X_full[tr],X_full[te]; ytr,yte=y[tr],y[te]; gtr=g_full[tr]
        for cvname,cv,grp in [('CV_random',StratifiedKFold(5,shuffle=True,random_state=SEED),None),
                              ('CV_group',GroupKFold(5),gtr)]:
            acc=[]
            for a,b in cv.split(Xtr,ytr,grp):
                if len(np.unique(ytr[b]))<2: continue
                sc=StandardScaler().fit(Xtr[a])
                mm=make_model(); mm.fit(sc.transform(Xtr[a]),ytr[a])
                prob=mm.predict_proba(sc.transform(Xtr[b]))[:,1] if hasattr(mm,'predict_proba') else mm.predict(sc.transform(Xtr[b]))
                acc.append(_metrics(ytr[b],prob))
            d=pd.DataFrame(acc).mean().to_dict()
            r={'model':name,'strain':s,'eval':cvname}
            for k in ['Accuracy','Precision','Recall','F1','AUC','MCC']: r[k]=round(d.get(k,np.nan),3)
            rows.append(r)
        sc=StandardScaler().fit(Xtr); mm=make_model(); mm.fit(sc.transform(Xtr),ytr)
        prob=mm.predict_proba(sc.transform(Xte))[:,1] if hasattr(mm,'predict_proba') else mm.predict(sc.transform(Xte))
        d=_metrics(yte,prob); r={'model':name,'strain':s,'eval':'independent_test'}
        for k in ['Accuracy','Precision','Recall','F1','AUC','MCC']: r[k]=round(d.get(k,np.nan),3)
        rows.append(r)
    df=pd.DataFrame(rows); df.to_csv(rf'{OUT}\train_{name}.csv',index=False)
    print(df.to_string(index=False)); return df

print('Cell nền OK: eval_sklearn, get_split, _metrics')

X: (303, 100000) | chủng: ['C4', 'C5', 'P9a']
Cell nền OK: eval_sklearn, get_split, _metrics


# 13. Machine Learning với SNP

13.1: LogReg

In [ ]:
from sklearn.linear_model import LogisticRegression
eval_sklearn('LogReg_SNP', lambda: LogisticRegression(
    C=0.05, max_iter=3000, class_weight='balanced', random_state=SEED))

     model strain             eval  Accuracy  Precision  Recall    F1   AUC   MCC
LogReg_SNP     C4        CV_random     0.703      0.688   0.701 0.694 0.759 0.405
LogReg_SNP     C4         CV_group     0.609      0.581   0.798 0.640 0.594 0.247
LogReg_SNP     C4 independent_test     0.705      0.688   0.733 0.710 0.754 0.411
LogReg_SNP     C5        CV_random     0.789      0.431   0.400 0.407 0.777 0.286
LogReg_SNP     C5         CV_group     0.771      0.215   0.148 0.157 0.600 0.069
LogReg_SNP     C5 independent_test     0.770      0.400   0.545 0.462 0.871 0.326
LogReg_SNP    P9a        CV_random     0.632      0.634   0.625 0.625 0.722 0.265
LogReg_SNP    P9a         CV_group     0.662      0.556   0.702 0.606 0.678 0.202
LogReg_SNP    P9a independent_test     0.738      0.675   0.900 0.771 0.808 0.506


,model,strain,eval,Accuracy,Precision,Recall,F1,AUC,MCC
0,LogReg_SNP,C4,CV_random,0.703,0.688,0.701,0.694,0.759,0.405
1,LogReg_SNP,C4,CV_group,0.609,0.581,0.798,0.640,0.594,0.247
2,LogReg_SNP,C4,independent_test,0.705,0.688,0.733,0.710,0.754,0.411
3,LogReg_SNP,C5,CV_random,0.789,0.431,0.400,0.407,0.777,0.286
4,LogReg_SNP,C5,CV_group,0.771,0.215,0.148,0.157,0.600,0.069
5,LogReg_SNP,C5,independent_test,0.770,0.400,0.545,0.462,0.871,0.326
6,LogReg_SNP,P9a,CV_random,0.632,0.634,0.625,0.625,0.722,0.265
7,LogReg_SNP,P9a,CV_group,0.662,0.556,0.702,0.606,0.678,0.202
8,LogReg_SNP,P9a,independent_test,0.738,0.675,0.900,0.771,0.808,0.506


13.2: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
eval_sklearn('RandomForest_SNP', lambda: RandomForestClassifier(
    n_estimators=300, max_depth=15, max_features='sqrt', min_samples_leaf=5,
    class_weight='balanced', n_jobs=-1, random_state=SEED))

           model strain             eval  Accuracy  Precision  Recall    F1   AUC   MCC
RandomForest_SNP     C4        CV_random     0.673      0.680   0.624 0.649 0.777 0.347
RandomForest_SNP     C4         CV_group     0.581      0.540   0.756 0.596 0.635 0.140
RandomForest_SNP     C4 independent_test     0.738      0.733   0.733 0.733 0.770 0.475
RandomForest_SNP     C5        CV_random     0.806      0.482   0.422 0.438 0.797 0.331
RandomForest_SNP     C5         CV_group     0.820      0.431   0.485 0.441 0.660 0.330
RandomForest_SNP     C5 independent_test     0.803      0.471   0.727 0.571 0.847 0.469
RandomForest_SNP    P9a        CV_random     0.752      0.728   0.800 0.761 0.793 0.508
RandomForest_SNP    P9a         CV_group     0.707      0.691   0.759 0.691 0.767 0.314
RandomForest_SNP    P9a independent_test     0.754      0.703   0.867 0.776 0.880 0.524


,model,strain,eval,Accuracy,Precision,Recall,F1,AUC,MCC
0,RandomForest_SNP,C4,CV_random,0.673,0.680,0.624,0.649,0.777,0.347
1,RandomForest_SNP,C4,CV_group,0.581,0.540,0.756,0.596,0.635,0.140
2,RandomForest_SNP,C4,independent_test,0.738,0.733,0.733,0.733,0.770,0.475
3,RandomForest_SNP,C5,CV_random,0.806,0.482,0.422,0.438,0.797,0.331
4,RandomForest_SNP,C5,CV_group,0.820,0.431,0.485,0.441,0.660,0.330
5,RandomForest_SNP,C5,independent_test,0.803,0.471,0.727,0.571,0.847,0.469
6,RandomForest_SNP,P9a,CV_random,0.752,0.728,0.800,0.761,0.793,0.508
7,RandomForest_SNP,P9a,CV_group,0.707,0.691,0.759,0.691,0.767,0.314
8,RandomForest_SNP,P9a,independent_test,0.754,0.703,0.867,0.776,0.880,0.524


13.3: XGBoost

In [ ]:
from xgboost import XGBClassifier
import numpy as np
spw = np.mean([(df_keep[s+'_bin']==0).sum()/max((df_keep[s+'_bin']==1).sum(),1)
               for s in ['C4','C5','P9a']])
eval_sklearn('XGBoost_SNP', lambda: XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.5, scale_pos_weight=spw, n_jobs=-1,
    random_state=SEED, verbosity=0, eval_metric='logloss'))

      model strain             eval  Accuracy  Precision  Recall    F1   AUC   MCC
XGBoost_SNP     C4        CV_random     0.694      0.682   0.691 0.686 0.779 0.389
XGBoost_SNP     C4         CV_group     0.600      0.549   0.836 0.641 0.661 0.191
XGBoost_SNP     C4 independent_test     0.770      0.750   0.800 0.774 0.796 0.543
XGBoost_SNP     C5        CV_random     0.826      0.530   0.378 0.429 0.792 0.347
XGBoost_SNP     C5         CV_group     0.783      0.264   0.184 0.215 0.657 0.089
XGBoost_SNP     C5 independent_test     0.836      0.545   0.545 0.545 0.882 0.445
XGBoost_SNP    P9a        CV_random     0.698      0.686   0.733 0.707 0.798 0.399
XGBoost_SNP    P9a         CV_group     0.716      0.699   0.801 0.718 0.726 0.332
XGBoost_SNP    P9a independent_test     0.738      0.684   0.867 0.765 0.867 0.495


,model,strain,eval,Accuracy,Precision,Recall,F1,AUC,MCC
0,XGBoost_SNP,C4,CV_random,0.694,0.682,0.691,0.686,0.779,0.389
1,XGBoost_SNP,C4,CV_group,0.600,0.549,0.836,0.641,0.661,0.191
2,XGBoost_SNP,C4,independent_test,0.770,0.750,0.800,0.774,0.796,0.543
3,XGBoost_SNP,C5,CV_random,0.826,0.530,0.378,0.429,0.792,0.347
4,XGBoost_SNP,C5,CV_group,0.783,0.264,0.184,0.215,0.657,0.089
5,XGBoost_SNP,C5,independent_test,0.836,0.545,0.545,0.545,0.882,0.445
6,XGBoost_SNP,P9a,CV_random,0.698,0.686,0.733,0.707,0.798,0.399
7,XGBoost_SNP,P9a,CV_group,0.716,0.699,0.801,0.718,0.726,0.332
8,XGBoost_SNP,P9a,independent_test,0.738,0.684,0.867,0.765,0.867,0.495


13.4: SVM

In [ ]:
from sklearn.svm import SVC
eval_sklearn('SVM_SNP', lambda: SVC(
    C=1.0, kernel='rbf', probability=True, class_weight='balanced', random_state=SEED))

  model strain             eval  Accuracy  Precision  Recall    F1   AUC   MCC
SVM_SNP     C4        CV_random     0.685      0.692   0.632 0.660 0.762 0.370
SVM_SNP     C4         CV_group     0.546      0.539   0.701 0.553 0.631 0.127
SVM_SNP     C4 independent_test     0.738      0.750   0.700 0.724 0.798 0.476
SVM_SNP     C5        CV_random     0.814      0.512   0.289 0.362 0.810 0.283
SVM_SNP     C5         CV_group     0.815      0.467   0.060 0.106 0.691 0.112
SVM_SNP     C5 independent_test     0.836      0.545   0.545 0.545 0.869 0.445
SVM_SNP    P9a        CV_random     0.677      0.675   0.683 0.678 0.753 0.356
SVM_SNP    P9a         CV_group     0.670      0.729   0.620 0.590 0.672 0.242
SVM_SNP    P9a independent_test     0.738      0.694   0.833 0.758 0.797 0.486


,model,strain,eval,Accuracy,Precision,Recall,F1,AUC,MCC
0,SVM_SNP,C4,CV_random,0.685,0.692,0.632,0.660,0.762,0.370
1,SVM_SNP,C4,CV_group,0.546,0.539,0.701,0.553,0.631,0.127
2,SVM_SNP,C4,independent_test,0.738,0.750,0.700,0.724,0.798,0.476
3,SVM_SNP,C5,CV_random,0.814,0.512,0.289,0.362,0.810,0.283
4,SVM_SNP,C5,CV_group,0.815,0.467,0.060,0.106,0.691,0.112
5,SVM_SNP,C5,independent_test,0.836,0.545,0.545,0.545,0.869,0.445
6,SVM_SNP,P9a,CV_random,0.677,0.675,0.683,0.678,0.753,0.356
7,SVM_SNP,P9a,CV_group,0.670,0.729,0.620,0.590,0.672,0.242
8,SVM_SNP,P9a,independent_test,0.738,0.694,0.833,0.758,0.797,0.486


# 14. Deeplearning với SNP

14.1: Định nghĩa và thiết lập các biến / yêu cầu

In [ ]:
# =====================================================================
# NỀN DEEP LEARNING cho SNP — harness lặp N lần + CV + test
# Chạy SAU cell nền (get_split, _metrics, BIN) và sau khi nạp X_full=X_snp
# =====================================================================
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import StandardScaler

dev='cuda' if torch.cuda.is_available() else 'cpu'
DL_REPEATS = 20          # số lần lặp (giảm để test nhanh)
DL_EPOCHS  = 60
print(f'Deep SNP | device: {dev} | X: {X_full.shape} | lặp: {DL_REPEATS}')

def fit_predict_dl(make_model, Xtr, ytr, Xte, epochs, seed):
    torch.manual_seed(seed)
    m=make_model().to(dev)
    Xt=torch.tensor(Xtr,dtype=torch.float32,device=dev); yt=torch.tensor(ytr,dtype=torch.float32,device=dev)
    Xe=torch.tensor(Xte,dtype=torch.float32,device=dev)
    npos=max(ytr.sum(),1); pw=torch.tensor([(len(ytr)-npos)/npos],dtype=torch.float32,device=dev)
    opt=torch.optim.Adam(m.parameters(),lr=1e-3,weight_decay=1e-3); lf=nn.BCEWithLogitsLoss(pos_weight=pw)
    m.train()
    for _ in range(epochs):
        opt.zero_grad(); lf(m(Xt).squeeze(-1),yt).backward(); opt.step()
    m.eval()
    with torch.no_grad(): p=torch.sigmoid(m(Xe).squeeze(-1)).cpu().numpy()
    del m, Xt, yt, Xe; torch.cuda.empty_cache()
    return p

def eval_deep(name, make_model, epochs=DL_EPOCHS, repeats=DL_REPEATS):
    rows=[]; t0=time.time()
    for s in BIN:
        tr,te,y=get_split(s)
        Xtr,Xte=X_full[tr],X_full[te]; ytr,yte=y[tr],y[te]; gtr=g_full[tr]
        for cvname,cv,grp in [('CV_random',StratifiedKFold(5,shuffle=True,random_state=SEED),None),
                              ('CV_group',GroupKFold(5),gtr)]:
            rep_m=[]
            for rep in range(repeats):
                fold=[]
                for a,b in cv.split(Xtr,ytr,grp):
                    if len(np.unique(ytr[b]))<2: continue
                    sc=StandardScaler().fit(Xtr[a])
                    prob=fit_predict_dl(make_model,sc.transform(Xtr[a]),ytr[a],sc.transform(Xtr[b]),epochs,rep)
                    fold.append(_metrics(ytr[b],prob))
                rep_m.append(pd.DataFrame(fold).mean())
            dm=pd.DataFrame(rep_m); r={'model':name,'strain':s,'eval':cvname}
            for k in ['Accuracy','Precision','Recall','F1','AUC','MCC']:
                r[k]=round(dm[k].mean(),3); r[k+'_std']=round(dm[k].std(),3)
            rows.append(r)
        # test độc lập
        tst=[]
        for rep in range(repeats):
            sc=StandardScaler().fit(Xtr)
            prob=fit_predict_dl(make_model,sc.transform(Xtr),ytr,sc.transform(Xte),epochs,rep)
            tst.append(_metrics(yte,prob))
        dm=pd.DataFrame(tst); r={'model':name,'strain':s,'eval':'independent_test'}
        for k in ['Accuracy','Precision','Recall','F1','AUC','MCC']:
            r[k]=round(dm[k].mean(),3); r[k+'_std']=round(dm[k].std(),3)
        rows.append(r)
    df=pd.DataFrame(rows); df.to_csv(rf'{OUT}\train_{name}.csv',index=False)
    cols=['model','strain','eval','Accuracy','F1','AUC','MCC']
    print(df[cols].to_string(index=False))
    print(f'\n{name} xong: {(time.time()-t0)/60:.1f} phút')
    return df

# ---------- KIẾN TRÚC ----------
N_IN = X_full.shape[1]   # 100000

class MLP_SNP(nn.Module):
    def __init__(s):
        super().__init__()
        s.net=nn.Sequential(nn.Linear(N_IN,256),nn.ReLU(),nn.BatchNorm1d(256),nn.Dropout(0.4),
                            nn.Linear(256,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x): return s.net(x)

class CNN1D_SNP(nn.Module):
    def __init__(s):
        super().__init__()
        s.conv=nn.Sequential(
            nn.Conv1d(1,16,11,stride=4,padding=5),nn.ReLU(),nn.MaxPool1d(4),
            nn.Conv1d(16,32,7,stride=2,padding=3),nn.ReLU(),nn.MaxPool1d(4),
            nn.AdaptiveAvgPool1d(8))
        s.head=nn.Sequential(nn.Flatten(),nn.Linear(32*8,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x): return s.head(s.conv(x.unsqueeze(1)))

class CNN2D_SNP(nn.Module):
    def __init__(s,h=250,w=400):   # 250*400=100000
        super().__init__(); s.h,s.w=h,w
        s.conv=nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4,4)))
        s.head=nn.Sequential(nn.Flatten(),nn.Linear(32*16,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x): return s.head(s.conv(x.view(x.size(0),1,s.h,s.w)))

class mCNN_SNP(nn.Module):  # multiple-window scanning + 1-max pool (mCNN-ETC)
    def __init__(s,kernels=(5,11,21),nfilt=32):
        super().__init__()
        s.br=nn.ModuleList([nn.Conv1d(1,nfilt,k,stride=2,padding=k//2) for k in kernels])
        s.head=nn.Sequential(nn.Linear(nfilt*len(kernels),64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x):
        x=x.unsqueeze(1); feats=[]
        for b in s.br:
            h=torch.relu(b(x)); h=h.max(dim=2).values   # 1-max pool
            feats.append(h)
        return s.head(torch.cat(feats,1))

print('Nền deep SNP OK: eval_deep + MLP_SNP, CNN1D_SNP, CNN2D_SNP, mCNN_SNP')

Deep SNP | device: cuda | X: (303, 100000) | lặp: 20
Nền deep SNP OK: eval_deep + MLP_SNP, CNN1D_SNP, CNN2D_SNP, mCNN_SNP


14.2: MLP với SNP

In [ ]:
BIN = ['C4','C5','P9a']            # đảm bảo đủ 3 chủng
eval_deep('MLP_SNP', MLP_SNP)

  model strain             eval  Accuracy    F1   AUC   MCC
MLP_SNP     C4        CV_random     0.699 0.688 0.780 0.401
MLP_SNP     C4         CV_group     0.553 0.595 0.635 0.138
MLP_SNP     C4 independent_test     0.735 0.735 0.763 0.472
MLP_SNP     C5        CV_random     0.824 0.542 0.809 0.439
MLP_SNP     C5         CV_group     0.795 0.241 0.677 0.146
MLP_SNP     C5 independent_test     0.787 0.516 0.871 0.399
MLP_SNP    P9a        CV_random     0.666 0.653 0.752 0.337
MLP_SNP    P9a         CV_group     0.668 0.608 0.689 0.171
MLP_SNP    P9a independent_test     0.729 0.743 0.799 0.464

MLP_SNP xong: 11.8 phút


,model,strain,eval,Accuracy,Accuracy_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,AUC,AUC_std,MCC,MCC_std
0,MLP_SNP,C4,CV_random,0.699,0.009,0.692,0.012,0.690,0.022,0.688,0.011,0.780,0.005,0.401,0.019
1,MLP_SNP,C4,CV_group,0.553,0.019,0.535,0.014,0.757,0.021,0.595,0.011,0.635,0.016,0.138,0.038
2,MLP_SNP,C4,independent_test,0.735,0.021,0.724,0.024,0.747,0.045,0.735,0.025,0.763,0.008,0.472,0.043
3,MLP_SNP,C5,CV_random,0.824,0.011,0.531,0.030,0.568,0.047,0.542,0.034,0.809,0.006,0.439,0.039
4,MLP_SNP,C5,CV_group,0.795,0.007,0.351,0.081,0.208,0.058,0.241,0.062,0.677,0.009,0.146,0.055
5,MLP_SNP,C5,independent_test,0.787,0.016,0.438,0.027,0.636,0.106,0.516,0.042,0.871,0.008,0.399,0.058
6,MLP_SNP,P9a,CV_random,0.666,0.009,0.686,0.009,0.630,0.018,0.653,0.011,0.752,0.006,0.337,0.017
7,MLP_SNP,P9a,CV_group,0.668,0.012,0.578,0.046,0.693,0.038,0.608,0.033,0.689,0.013,0.171,0.044
8,MLP_SNP,P9a,independent_test,0.729,0.017,0.695,0.014,0.798,0.035,0.743,0.019,0.799,0.010,0.464,0.037


14.3: CNN-1D với SNP

In [ ]:
BIN = ['C4','C5','P9a']            # đảm bảo đủ 3 chủng
eval_deep('CNN1D_SNP', CNN1D_SNP)

    model strain             eval  Accuracy    F1   AUC   MCC
CNN1D_SNP     C4        CV_random     0.602 0.491 0.635 0.204
CNN1D_SNP     C4         CV_group     0.565 0.450 0.506 0.087
CNN1D_SNP     C4 independent_test     0.678 0.650 0.706 0.357
CNN1D_SNP     C5        CV_random     0.734 0.449 0.718 0.300
CNN1D_SNP     C5         CV_group     0.729 0.383 0.700 0.194
CNN1D_SNP     C5 independent_test     0.689 0.454 0.731 0.316
CNN1D_SNP    P9a        CV_random     0.590 0.493 0.687 0.195
CNN1D_SNP    P9a         CV_group     0.585 0.470 0.587 0.134
CNN1D_SNP    P9a independent_test     0.610 0.565 0.666 0.223

CNN1D_SNP xong: 21.0 phút


,model,strain,eval,Accuracy,Accuracy_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,AUC,AUC_std,MCC,MCC_std
0,CNN1D_SNP,C4,CV_random,0.602,0.005,0.633,0.009,0.403,0.012,0.491,0.009,0.635,0.006,0.204,0.011
1,CNN1D_SNP,C4,CV_group,0.565,0.020,0.587,0.024,0.441,0.011,0.450,0.015,0.506,0.025,0.087,0.037
2,CNN1D_SNP,C4,independent_test,0.678,0.011,0.698,0.014,0.608,0.015,0.650,0.012,0.706,0.010,0.357,0.022
3,CNN1D_SNP,C5,CV_random,0.734,0.003,0.371,0.005,0.578,0.000,0.449,0.003,0.718,0.006,0.300,0.004
4,CNN1D_SNP,C5,CV_group,0.729,0.006,0.304,0.008,0.539,0.018,0.383,0.009,0.700,0.008,0.194,0.012
5,CNN1D_SNP,C5,independent_test,0.689,0.018,0.332,0.018,0.718,0.028,0.454,0.021,0.731,0.007,0.316,0.030
6,CNN1D_SNP,P9a,CV_random,0.590,0.011,0.643,0.016,0.414,0.031,0.493,0.025,0.687,0.004,0.195,0.022
7,CNN1D_SNP,P9a,CV_group,0.585,0.021,0.617,0.044,0.451,0.041,0.470,0.028,0.587,0.013,0.134,0.024
8,CNN1D_SNP,P9a,independent_test,0.610,0.020,0.629,0.038,0.518,0.059,0.565,0.030,0.666,0.009,0.223,0.042


14.4: CNN-2D với SNP

In [ ]:
BIN = ['C4','C5','P9a']            # đảm bảo đủ 3 chủng
eval_deep('CNN2D_SNP', CNN2D_SNP)

    model strain             eval  Accuracy    F1   AUC   MCC
CNN2D_SNP     C4        CV_random     0.616 0.536 0.669 0.230
CNN2D_SNP     C4         CV_group     0.594 0.527 0.558 0.131
CNN2D_SNP     C4 independent_test     0.655 0.650 0.716 0.312
CNN2D_SNP     C5        CV_random     0.737 0.450 0.722 0.303
CNN2D_SNP     C5         CV_group     0.745 0.376 0.669 0.218
CNN2D_SNP     C5 independent_test     0.693 0.461 0.756 0.325
CNN2D_SNP    P9a        CV_random     0.605 0.518 0.689 0.229
CNN2D_SNP    P9a         CV_group     0.603 0.441 0.563 0.071
CNN2D_SNP    P9a independent_test     0.660 0.586 0.707 0.334

CNN2D_SNP xong: 85.0 phút


,model,strain,eval,Accuracy,Accuracy_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,AUC,AUC_std,MCC,MCC_std
0,CNN2D_SNP,C4,CV_random,0.616,0.012,0.631,0.014,0.473,0.053,0.536,0.035,0.669,0.012,0.230,0.025
1,CNN2D_SNP,C4,CV_group,0.594,0.015,0.598,0.020,0.497,0.052,0.527,0.041,0.558,0.009,0.131,0.029
2,CNN2D_SNP,C4,independent_test,0.655,0.030,0.654,0.047,0.650,0.044,0.650,0.020,0.716,0.016,0.312,0.060
3,CNN2D_SNP,C5,CV_random,0.737,0.003,0.375,0.007,0.573,0.009,0.450,0.008,0.722,0.006,0.303,0.009
4,CNN2D_SNP,C5,CV_group,0.745,0.012,0.313,0.011,0.508,0.000,0.376,0.008,0.669,0.012,0.218,0.021
5,CNN2D_SNP,C5,independent_test,0.693,0.009,0.337,0.008,0.727,0.000,0.461,0.008,0.756,0.009,0.325,0.010
6,CNN2D_SNP,P9a,CV_random,0.605,0.018,0.669,0.017,0.446,0.074,0.518,0.055,0.689,0.010,0.229,0.032
7,CNN2D_SNP,P9a,CV_group,0.603,0.014,0.524,0.049,0.409,0.062,0.441,0.051,0.563,0.014,0.071,0.026
8,CNN2D_SNP,P9a,independent_test,0.660,0.032,0.725,0.038,0.502,0.102,0.586,0.070,0.707,0.014,0.334,0.059


14.5: mCNN với SNP

In [ ]:
BIN = ['C4','C5','P9a']            # đảm bảo đủ 3 chủng
eval_deep('mCNN_SNP', mCNN_SNP)

   model strain             eval  Accuracy    F1   AUC   MCC
mCNN_SNP     C4        CV_random     0.623 0.556 0.687 0.253
mCNN_SNP     C4         CV_group     0.565 0.457 0.575 0.095
mCNN_SNP     C4 independent_test     0.653 0.599 0.713 0.320
mCNN_SNP     C5        CV_random     0.757 0.402 0.720 0.272
mCNN_SNP     C5         CV_group     0.749 0.362 0.664 0.210
mCNN_SNP     C5 independent_test     0.743 0.494 0.776 0.370
mCNN_SNP    P9a        CV_random     0.604 0.522 0.684 0.229
mCNN_SNP    P9a         CV_group     0.587 0.444 0.597 0.128
mCNN_SNP    P9a independent_test     0.639 0.602 0.723 0.292

mCNN_SNP xong: 103.9 phút


,model,strain,eval,Accuracy,Accuracy_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,AUC,AUC_std,MCC,MCC_std
0,mCNN_SNP,C4,CV_random,0.623,0.022,0.646,0.040,0.514,0.061,0.556,0.035,0.687,0.022,0.253,0.045
1,mCNN_SNP,C4,CV_group,0.565,0.047,0.546,0.095,0.503,0.079,0.457,0.066,0.575,0.032,0.095,0.060
2,mCNN_SNP,C4,independent_test,0.653,0.035,0.700,0.069,0.548,0.142,0.599,0.085,0.713,0.029,0.320,0.064
3,mCNN_SNP,C5,CV_random,0.757,0.028,0.392,0.034,0.460,0.064,0.402,0.024,0.720,0.024,0.272,0.026
4,mCNN_SNP,C5,CV_group,0.749,0.029,0.338,0.063,0.471,0.108,0.362,0.070,0.664,0.025,0.210,0.072
5,mCNN_SNP,C5,independent_test,0.743,0.041,0.389,0.044,0.691,0.085,0.494,0.036,0.776,0.032,0.370,0.051
6,mCNN_SNP,P9a,CV_random,0.604,0.024,0.659,0.036,0.478,0.100,0.522,0.075,0.684,0.025,0.229,0.050
7,mCNN_SNP,P9a,CV_group,0.587,0.021,0.658,0.090,0.416,0.056,0.444,0.039,0.597,0.033,0.128,0.061
8,mCNN_SNP,P9a,independent_test,0.639,0.048,0.661,0.067,0.590,0.183,0.602,0.104,0.723,0.047,0.292,0.091


# 15. Lưu models SNP

In [ ]:
# =====================================================================
# LƯU MODEL SNP (deep + ML) + artifact tái lập cho teammate
# Chạy trên máy thuê (Windows) SAU khi train xong. Cần X_full=X_snp trong RAM.
# =====================================================================
import os, json, joblib, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler

OUT = r'C:\GenBBPipeline\Export'
MODEL_DIR = rf'{OUT}\models_snp'
os.makedirs(MODEL_DIR, exist_ok=True)
dev='cuda' if torch.cuda.is_available() else 'cpu'
BINS=[s for s in ['C4','C5','P9a'] if s+'_bin' in df_keep.columns]
N_IN=X_full.shape[1]
manifest=[]

# ---- LƯU snp_idx (100k SNP đã chọn) — QUAN TRỌNG NHẤT cho tái lập ----
z=np.load(rf'{OUT}\X_snp_genome.npz', allow_pickle=True)
snp_idx=z['snp_idx']
np.save(rf'{MODEL_DIR}\snp_idx_100k.npy', snp_idx)
print(f'Lưu snp_idx: {len(snp_idx)} SNP (index trong file .bim gốc)')

# ---- LƯU ML models (nếu đã train, train lại trên full để lưu) ----
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
spw=np.mean([(df_keep[s+'_bin']==0).sum()/max((df_keep[s+'_bin']==1).sum(),1) for s in BINS])
SK={'LogReg':lambda:LogisticRegression(C=0.05,max_iter=3000,class_weight='balanced',random_state=SEED),
    'RandomForest':lambda:RandomForestClassifier(n_estimators=300,max_depth=15,max_features='sqrt',
        min_samples_leaf=5,class_weight='balanced',n_jobs=-1,random_state=SEED),
    'SVM':lambda:SVC(C=1.0,kernel='rbf',probability=True,class_weight='balanced',random_state=SEED)}
try:
    from xgboost import XGBClassifier
    SK['XGBoost']=lambda:XGBClassifier(n_estimators=300,max_depth=4,learning_rate=0.05,subsample=0.8,
        colsample_bytree=0.5,scale_pos_weight=spw,n_jobs=-1,random_state=SEED,verbosity=0,eval_metric='logloss')
except: pass

print('=== Lưu ML models SNP ===')
for s in BINS:
    y=df_keep[s+'_bin'].values.astype(int)
    sc=StandardScaler().fit(X_full)
    for name,mk in SK.items():
        m=mk(); m.fit(sc.transform(X_full),y)
        fn=rf'{MODEL_DIR}\{name}_SNP_{s}.joblib'
        joblib.dump({'model':m,'scaler':sc,'strain':s,'n_features':N_IN,
                     'feature':'X_snp_100k','snp_idx_file':'snp_idx_100k.npy'}, fn)
        manifest.append(dict(file=os.path.basename(fn),model=name,strain=s,framework='sklearn'))
    print(f'  {s}: {list(SK.keys())}')

# ---- LƯU DEEP models (.pt + arch) ----
class MLP_SNP(nn.Module):
    def __init__(s):
        super().__init__()
        s.net=nn.Sequential(nn.Linear(N_IN,256),nn.ReLU(),nn.BatchNorm1d(256),nn.Dropout(0.4),
                            nn.Linear(256,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x): return s.net(x)
class CNN1D_SNP(nn.Module):
    def __init__(s):
        super().__init__()
        s.conv=nn.Sequential(nn.Conv1d(1,16,11,stride=4,padding=5),nn.ReLU(),nn.MaxPool1d(4),
            nn.Conv1d(16,32,7,stride=2,padding=3),nn.ReLU(),nn.MaxPool1d(4),nn.AdaptiveAvgPool1d(8))
        s.head=nn.Sequential(nn.Flatten(),nn.Linear(32*8,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x): return s.head(s.conv(x.unsqueeze(1)))
class CNN2D_SNP(nn.Module):
    def __init__(s,h=250,w=400):
        super().__init__(); s.h,s.w=h,w
        s.conv=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.AdaptiveAvgPool2d((4,4)))
        s.head=nn.Sequential(nn.Flatten(),nn.Linear(32*16,64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x): return s.head(s.conv(x.view(x.size(0),1,s.h,s.w)))
class mCNN_SNP(nn.Module):
    def __init__(s,kernels=(5,11,21),nfilt=32):
        super().__init__()
        s.br=nn.ModuleList([nn.Conv1d(1,nfilt,k,stride=2,padding=k//2) for k in kernels])
        s.head=nn.Sequential(nn.Linear(nfilt*len(kernels),64),nn.ReLU(),nn.Dropout(0.3),nn.Linear(64,1))
    def forward(s,x):
        x=x.unsqueeze(1); feats=[torch.relu(b(x)).max(dim=2).values for b in s.br]
        return s.head(torch.cat(feats,1))

DEEP={'MLP_SNP':MLP_SNP,'CNN1D_SNP':CNN1D_SNP,'CNN2D_SNP':CNN2D_SNP,'mCNN_SNP':mCNN_SNP}
print('=== Lưu DEEP models SNP (train full + save) ===')
for s in BINS:
    y=df_keep[s+'_bin'].values.astype(int)
    sc=StandardScaler().fit(X_full)
    Xt=torch.tensor(sc.transform(X_full),dtype=torch.float32,device=dev)
    yt=torch.tensor(y,dtype=torch.float32,device=dev)
    npos=max(y.sum(),1); pw=torch.tensor([(len(y)-npos)/npos],dtype=torch.float32,device=dev)
    for name,cls in DEEP.items():
        torch.manual_seed(SEED); m=cls().to(dev); m.train()
        opt=torch.optim.Adam(m.parameters(),lr=1e-3,weight_decay=1e-3); lf=nn.BCEWithLogitsLoss(pos_weight=pw)
        for _ in range(60): opt.zero_grad(); lf(m(Xt).squeeze(-1),yt).backward(); opt.step()
        torch.save({'state_dict':m.state_dict(),'arch':name,'n_in':N_IN,'strain':s}, rf'{MODEL_DIR}\{name}_{s}.pt')
        joblib.dump(sc, rf'{MODEL_DIR}\{name}_{s}_scaler.joblib')
        manifest.append(dict(file=f'{name}_{s}.pt',model=name,strain=s,framework='torch'))
    print(f'  {s}: {list(DEEP.keys())}')

pd.DataFrame(manifest).to_csv(rf'{MODEL_DIR}\manifest.csv',index=False)

# ---- README/spec cho nhánh SNP ----
readme=f"""# MODELS SNP (toàn genome, 100k) — spec tái lập

## ĐẶC TRƯNG: X_snp 100.000 chiều
Nguồn: PLINK genotype base_filtered_v0.7 (3024 mẫu × 4.82M SNP)

## PREPROCESSING (tái lập chính xác)
1. Lọc {len(df_keep)} mẫu khớp df_keep (theo ID, ví dụ B001..)
2. MAF filter > 0.05 (tính trên {len(df_keep)} mẫu, MAF=min(p,1-p), p=mean(genotype)/2)
3. Chọn 100.000 SNP top-variance (trong nhóm qua MAF)
4. ⚠️ SNP CHÍNH XÁC: dùng snp_idx_100k.npy (index cột trong file .bim gốc)
   -> teammate PHẢI dùng đúng file này để lấy đúng 100k SNP, KHÔNG tự chọn lại
5. Điền NaN (thiếu) bằng trung bình cột
6. StandardScaler (kèm mỗi model .joblib / _scaler.joblib)

## FILE
- snp_idx_100k.npy      : index 100k SNP đã chọn (BẮT BUỘC để tái lập)
- {{model}}_SNP_{{strain}}.joblib : ML models (LogReg/RF/XGBoost/SVM) + scaler
- {{model}}_{{strain}}.pt          : deep models (MLP/CNN1D/CNN2D/mCNN) state_dict
- {{model}}_{{strain}}_scaler.joblib : scaler cho deep
- manifest.csv

## LOAD deep (torch)
    ck=torch.load('CNN1D_SNP_P9a.pt')  # ck['arch'], ck['n_in'], ck['state_dict']
    # tái tạo class theo ck['arch'] (định nghĩa trong notebook), load_state_dict

## LƯU Ý
- Kết quả: xem results tổng hợp. So với nhánh embedding để đối chiếu.
- Model minh họa pipeline, không phải predictor chính xác cao.
"""
open(rf'{MODEL_DIR}\README_snp.txt','w',encoding='utf-8').write(readme)

import shutil
shutil.make_archive(rf'{OUT}\models_snp','zip',MODEL_DIR)
print(f'\n=== XONG. {len(manifest)} model + snp_idx ===')
print(f'>>> Gửi teammate: {OUT}\\models_snp.zip <<<')
print('⚠️ NHỚ TẢI ZIP VỀ trước khi trả máy thuê!')

Lưu snp_idx: 100000 SNP (index trong file .bim gốc)
=== Lưu ML models SNP ===
  C4: ['LogReg', 'RandomForest', 'SVM', 'XGBoost']
  C5: ['LogReg', 'RandomForest', 'SVM', 'XGBoost']
  P9a: ['LogReg', 'RandomForest', 'SVM', 'XGBoost']
=== Lưu DEEP models SNP (train full + save) ===
  C4: ['MLP_SNP', 'CNN1D_SNP', 'CNN2D_SNP', 'mCNN_SNP']
  C5: ['MLP_SNP', 'CNN1D_SNP', 'CNN2D_SNP', 'mCNN_SNP']
  P9a: ['MLP_SNP', 'CNN1D_SNP', 'CNN2D_SNP', 'mCNN_SNP']

=== XONG. 24 model + snp_idx ===
>>> Gửi teammate: C:\GenBBPipeline\Export\models_snp.zip <<<
⚠️ NHỚ TẢI ZIP VỀ trước khi trả máy thuê!


15.1: Xem các file models của embedding và SNP trước khi tổng hợp so sánh

In [ ]:
import glob, os
OUT = r'C:\GenBBPipeline\Export'
files = sorted(glob.glob(rf'{OUT}\train_*.csv'))
print(f'Có {len(files)} file:')
for f in files: print('  ', os.path.basename(f))

Có 17 file:
   train_CNN1D_SNP.csv
   train_CNN1D_emb.csv
   train_CNN2D_SNP.csv
   train_CNN2D_emb.csv
   train_LogReg.csv
   train_LogReg_SNP.csv
   train_MLP.csv
   train_MLP_SNP.csv
   train_RandomForest.csv
   train_RandomForest_SNP.csv
   train_SMOKE.csv
   train_SVM.csv
   train_SVM_SNP.csv
   train_XGBoost.csv
   train_XGBoost_SNP.csv
   train_mCNN_SNP.csv
   train_mCNN_emb.csv


15.2: Tạo bảng tổng hợp các models embedding và SNP

In [ ]:
# =====================================================================
# TỔNG HỢP 2 NHÁNH: SNP (toàn genome) vs EMBEDDING (vùng gene)
# Ma trận so sánh {SNP, embedding} × {classical, deep}
# =====================================================================
import pandas as pd, glob, os, numpy as np

OUT = r'C:\GenBBPipeline\Export'
files = [f for f in glob.glob(rf'{OUT}\train_*.csv') if 'SMOKE' not in f]

# gộp tất cả, gắn nhãn nhánh + loại model
parts=[]
for f in files:
    name=os.path.basename(f).replace('train_','').replace('.csv','')
    d=pd.read_csv(f)
    keep=[c for c in d.columns if not c.endswith('_std')]
    d=d[keep].copy()
    d['branch']='SNP' if name.endswith('_SNP') else 'embedding'
    base=name.replace('_SNP','').replace('_emb','')
    d['family']='deep' if base in ['MLP','CNN1D','CNN2D','mCNN'] else 'classical'
    d['model_base']=base
    parts.append(d)
RES=pd.concat(parts,ignore_index=True)
RES.to_csv(rf'{OUT}\results_ALL_2branch.csv',index=False)
print(f'Gộp {len(files)} file | {len(RES)} dòng\n')

# ===== BẢNG 1: CV_group MCC trung bình theo nhánh × loại =====
print('='*70); print('BẢNG 1 — CV_group MCC trung bình: nhánh × loại model'); print('='*70)
g=RES[RES['eval']=='CV_group']
piv=g.pivot_table(index='branch',columns='family',values='MCC',aggfunc='mean').round(3)
print(piv.to_string())

# ===== BẢNG 2: từng model, CV_group MCC trung bình 3 chủng =====
print('\n'+'='*70); print('BẢNG 2 — CV_group MCC mỗi model (TB 3 chủng), xếp hạng'); print('='*70)
m=g.groupby(['branch','model_base']).MCC.mean().round(3).sort_values(ascending=False)
print(m.to_string())

# ===== BẢNG 3: independent_test AUC, nhánh × loại =====
print('\n'+'='*70); print('BẢNG 3 — Independent test AUC trung bình: nhánh × loại'); print('='*70)
t=RES[RES['eval']=='independent_test']
pivt=t.pivot_table(index='branch',columns='family',values='AUC',aggfunc='mean').round(3)
print(pivt.to_string())

# ===== BẢNG 4: rò rỉ random vs group theo nhánh =====
print('\n'+'='*70); print('BẢNG 4 — Rò rỉ (random-group AUC) theo nhánh'); print('='*70)
leak=RES[RES['eval'].isin(['CV_random','CV_group'])].groupby(['branch','eval']).AUC.mean().unstack().round(3)
leak['gap']=(leak['CV_random']-leak['CV_group']).round(3)
print(leak.to_string())

# ===== BẢNG 5: top model mỗi nhánh (theo test AUC) =====
print('\n'+'='*70); print('BẢNG 5 — Model tốt nhất mỗi nhánh (independent_test, TB 3 chủng)'); print('='*70)
best=t.groupby(['branch','model_base'])[['AUC','MCC','F1']].mean().round(3)
for br in ['SNP','embedding']:
    print(f'\n--- {br} ---')
    print(best.loc[br].sort_values('AUC',ascending=False).to_string())

print(f'\nĐã lưu results_ALL_2branch.csv')

Gộp 16 file | 144 dòng

BẢNG 1 — CV_group MCC trung bình: nhánh × loại model
family     classical   deep
branch                     
SNP            0.200  0.144
embedding      0.183  0.149

BẢNG 2 — CV_group MCC mỗi model (TB 3 chủng), xếp hạng
branch     model_base  
SNP        RandomForest    0.261
embedding  RandomForest    0.218
           XGBoost         0.204
SNP        XGBoost         0.204
embedding  MLP             0.185
           CNN2D           0.185
SNP        LogReg          0.173
embedding  CNN1D           0.169
SNP        SVM             0.160
embedding  LogReg          0.158
           SVM             0.153
SNP        MLP             0.152
           mCNN            0.144
           CNN2D           0.140
           CNN1D           0.138
embedding  mCNN            0.057

BẢNG 3 — Independent test AUC trung bình: nhánh × loại
family     classical   deep
branch                     
SNP            0.828  0.744
embedding      0.758  0.686

BẢNG 4 — Rò rỉ (random-group AUC) 

# 16. Machine Learning trên DNA thô

Kiểm tra các files trước khi train

In [46]:
import glob, numpy as np
OUT = r'C:\GenBBPipeline\Export'
files = glob.glob(rf'{OUT}\regions_seq\*.npz')
print(f'regions_seq: {len(files)} file')
# xem độ dài các gene ở 1 mẫu
if files:
    z = np.load(files[0], allow_pickle=True)
    for k in z.files:
        print(f'  {k}: {len(str(z[k]))} bp')

regions_seq: 327 file
  Xa1: 9117 bp
  Xa21: 7791 bp
  Xa3_Xa26: 8517 bp
  xa5: 11926 bp
  xa13_SWEET11: 8553 bp
  xa25_SWEET13: 8899 bp


16.1: Thiết lập các thông số

In [52]:
# =====================================================================
# NHÁNH DNA THÔ — dựng X_raw one-hot từ regions_seq (5 gene)
# Mỗi gene cắt/pad về MAXLEN_GENE, one-hot 4 kênh (A/C/G/T), nối 5 gene.
# X_raw shape: (N, 4, 5*MAXLEN_GENE)
# =====================================================================
import os, glob, numpy as np, pandas as pd

OUT = r'C:\GenBBPipeline\Export'
GENE_RAW = ['xa5','xa13_SWEET11','xa25_SWEET13','Xa21','Xa1']  # KHỚP embedding
MAXLEN_GENE = 3000     # cắt/pad mỗi gene (promoter+gene start; đủ vùng tín hiệu)
BASE = {'A':0,'C':1,'G':2,'T':3}

df_raw = pd.read_pickle(rf'{OUT}\df_keep_14locus.pkl')
ids = list(df_raw['id'].astype(str))
print(f'{len(ids)} mẫu | gene: {GENE_RAW} | MAXLEN_GENE={MAXLEN_GENE}')

def onehot(seq, L):
    seq = seq.upper()[:L]
    m = np.zeros((4, L), dtype='float32')
    for i,ch in enumerate(seq):
        j = BASE.get(ch)
        if j is not None: m[j,i] = 1.0
    return m

rows = []
n_missing = {g:0 for g in GENE_RAW}
for a in ids:
    f = rf'{OUT}\regions_seq\{a}.npz'
    parts = []
    if os.path.exists(f):
        z = np.load(f, allow_pickle=True)
        for g in GENE_RAW:
            if g in z.files:
                parts.append(onehot(str(z[g]), MAXLEN_GENE))
            else:
                parts.append(np.zeros((4,MAXLEN_GENE),dtype='float32')); n_missing[g]+=1
    else:
        for g in GENE_RAW: parts.append(np.zeros((4,MAXLEN_GENE),dtype='float32')); n_missing[g]+=1
    rows.append(np.concatenate(parts, axis=1))    # (4, 5*MAXLEN_GENE)

X_raw = np.stack(rows).astype('float32')          # (N, 4, 5*MAXLEN)
df_raw = df_raw.reset_index(drop=True)
g_raw = df_raw['subgroup'].values
print(f'X_raw: {X_raw.shape}  (N, 4 kênh, {5*MAXLEN_GENE} bp)')
print('Gene thiếu (pad 0):', {k:v for k,v in n_missing.items() if v>0})

np.savez_compressed(rf'{OUT}\X_raw_dna.npz', X_raw=X_raw, ids=np.array(ids))
df_raw.to_pickle(rf'{OUT}\df_raw_keep.pkl')
print('Đã lưu X_raw_dna.npz + df_raw_keep.pkl')

303 mẫu | gene: ['xa5', 'xa13_SWEET11', 'xa25_SWEET13', 'Xa21', 'Xa1'] | MAXLEN_GENE=3000
X_raw: (303, 4, 15000)  (N, 4 kênh, 15000 bp)
Gene thiếu (pad 0): {'Xa1': 62}
Đã lưu X_raw_dna.npz + df_raw_keep.pkl


In [53]:
# =====================================================================
# HARNESS DEEP cho DNA THÔ (one-hot) — mini-batch, KHÔNG chuẩn hóa
# eval_deep_raw(name, model, X, df, g, out_dir): X là (N,4,L)
# CNN1D_raw, CNN2D_raw, mCNN_ETC (kiến trúc gốc paper thầy cho one-hot)
# =====================================================================
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, time
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,matthews_corrcoef)

dev='cuda' if torch.cuda.is_available() else 'cpu'
DL_REPEATS=20; DL_EPOCHS=60; BATCH=16
print(f'Deep RAW | device:{dev} | lặp:{DL_REPEATS} | batch:{BATCH}')

def _metrics(yt,prob,thr=0.5):
    yp=(prob>=thr).astype(int)
    d=dict(Accuracy=accuracy_score(yt,yp),Precision=precision_score(yt,yp,zero_division=0),
           Recall=recall_score(yt,yp,zero_division=0),F1=f1_score(yt,yp,zero_division=0),MCC=matthews_corrcoef(yt,yp))
    try: d['AUC']=roc_auc_score(yt,prob)
    except: d['AUC']=np.nan
    return d

def fit_predict_raw(make_model, Xtr, ytr, Xte, epochs, seed):
    torch.manual_seed(seed)
    m=make_model().to(dev)
    npos=max(ytr.sum(),1); pw=torch.tensor([(len(ytr)-npos)/npos],dtype=torch.float32,device=dev)
    opt=torch.optim.Adam(m.parameters(),lr=1e-3,weight_decay=1e-3); lf=nn.BCEWithLogitsLoss(pos_weight=pw)
    Xt=torch.tensor(Xtr,dtype=torch.float32); yt=torch.tensor(ytr,dtype=torch.float32)
    m.train()
    for _ in range(epochs):
        perm=torch.randperm(len(Xt))
        for i in range(0,len(Xt),BATCH):
            idx=perm[i:i+BATCH]
            xb=Xt[idx].to(dev); yb=yt[idx].to(dev)
            opt.zero_grad(); lf(m(xb).squeeze(-1),yb).backward(); opt.step()
    m.eval(); probs=[]
    Xe=torch.tensor(Xte,dtype=torch.float32)
    with torch.no_grad():
        for i in range(0,len(Xe),BATCH):
            xb=Xe[i:i+BATCH].to(dev)
            probs.append(torch.sigmoid(m(xb).squeeze(-1)).cpu().numpy())
    del m; torch.cuda.empty_cache()
    return np.concatenate(probs)

def eval_deep_raw(name, make_model, X, df, g, out_dir, epochs=DL_EPOCHS, repeats=DL_REPEATS, seed=42):
    BIN=[s for s in ['C4','C5','P9a'] if s+'_bin' in df.columns]
    rows=[]; t0=time.time()
    for s in BIN:
        y=df[s+'_bin'].values.astype(int); idx=np.arange(len(y))
        tr,te=train_test_split(idx,test_size=0.20,stratify=y,random_state=seed)
        Xtr,Xte=X[tr],X[te]; ytr,yte=y[tr],y[te]; gtr=g[tr]
        for cvname,cv,grp in [('CV_random',StratifiedKFold(5,shuffle=True,random_state=seed),None),
                              ('CV_group',GroupKFold(5),gtr)]:
            rep_m=[]
            for rep in range(repeats):
                fold=[]
                for a,b in cv.split(Xtr,ytr,grp):
                    if len(np.unique(ytr[b]))<2: continue
                    prob=fit_predict_raw(make_model,Xtr[a],ytr[a],Xtr[b],epochs,rep)
                    fold.append(_metrics(ytr[b],prob))
                rep_m.append(pd.DataFrame(fold).mean())
            dm=pd.DataFrame(rep_m); r={'model':name,'strain':s,'eval':cvname}
            for k in ['Accuracy','Precision','Recall','F1','AUC','MCC']:
                r[k]=round(dm[k].mean(),3); r[k+'_std']=round(dm[k].std(),3)
            rows.append(r)
        tst=[]
        for rep in range(repeats):
            prob=fit_predict_raw(make_model,Xtr,ytr,Xte,epochs,rep); tst.append(_metrics(yte,prob))
        dm=pd.DataFrame(tst); r={'model':name,'strain':s,'eval':'independent_test'}
        for k in ['Accuracy','Precision','Recall','F1','AUC','MCC']:
            r[k]=round(dm[k].mean(),3); r[k+'_std']=round(dm[k].std(),3)
        rows.append(r)
    df_res=pd.DataFrame(rows); df_res.to_csv(rf'{out_dir}\train_{name}.csv',index=False)
    print(df_res[['model','strain','eval','Accuracy','F1','AUC','MCC']].to_string(index=False))
    print(f'\n{name} xong: {(time.time()-t0)/60:.1f} phút'); return df_res

# ---------- KIẾN TRÚC one-hot (N,4,L) ----------
class CNN1D_raw(nn.Module):
    def __init__(s):
        super().__init__()
        s.conv=nn.Sequential(
            nn.Conv1d(4,32,15,stride=4,padding=7),nn.ReLU(),nn.MaxPool1d(4),
            nn.Conv1d(32,64,9,stride=2,padding=4),nn.ReLU(),nn.MaxPool1d(4),
            nn.Conv1d(64,64,5,padding=2),nn.ReLU(),nn.AdaptiveAvgPool1d(8))
        s.head=nn.Sequential(nn.Flatten(),nn.Linear(64*8,64),nn.ReLU(),nn.Dropout(0.4),nn.Linear(64,1))
    def forward(s,x): return s.head(s.conv(x))

class CNN2D_raw(nn.Module):
    # (N,4,L) -> ảnh (N,1,4,L)
    def __init__(s):
        super().__init__()
        s.conv=nn.Sequential(
            nn.Conv2d(1,16,(4,15),stride=(1,4),padding=(0,7)),nn.ReLU(),nn.MaxPool2d((1,4)),
            nn.Conv2d(16,32,(1,9),stride=(1,2),padding=(0,4)),nn.ReLU(),nn.AdaptiveAvgPool2d((1,8)))
        s.head=nn.Sequential(nn.Flatten(),nn.Linear(32*8,64),nn.ReLU(),nn.Dropout(0.4),nn.Linear(64,1))
    def forward(s,x): return s.head(s.conv(x.unsqueeze(1)))

# =====================================================================
# mCNN — KHỚP ĐÚNG FORMAT code gốc paper thầy (chuyển TF->PyTorch)
# Input gốc: (max_length, num_feature); conv kernel (1, window) quét dọc sequence.
# Cho DNA: num_feature=4 (A/C/G/T). Input tensor: (N, 1, max_length, 4)
# =====================================================================
import torch, torch.nn as nn, torch.nn.functional as F

class mCNN(nn.Module):
    def __init__(self, window_sizes=(8,24,48), max_length=15000,   # 3 cửa sổ, 15000
                 num_feature=4, num_filters=128, num_hidden=512, num_class=1):
        super().__init__()
        self.window_sizes=window_sizes
        self.max_length=max_length
        self.num_feature=num_feature
        self.convs=nn.ModuleList([
            nn.Conv2d(1, num_filters, kernel_size=(w, num_feature)) for w in window_sizes
        ])
        self.dropout=nn.Dropout(0.7)
        self.fc1=nn.Linear(num_filters*len(window_sizes), num_hidden)
        self.fc2=nn.Linear(num_hidden, num_class)

    def forward(self, x):
        if x.dim()==3:
            x=x.transpose(1,2).unsqueeze(1)         # (N,4,L)->(N,1,L,4)
        feats=[]
        for conv in self.convs:
            c=F.relu(conv(x))
            p=c.max(dim=2).values.squeeze(-1)       # 1-max pool
            feats.append(p)
        h=self.dropout(torch.cat(feats,1))
        return self.fc2(F.relu(self.fc1(h)))

print('mCNN nhẹ OK: 3 cửa sổ (8,24,48), max_length=15000')

Deep RAW | device:cuda | lặp:20 | batch:16
mCNN nhẹ OK: 3 cửa sổ (8,24,48), max_length=15000


16.2: CNN-1D với DNA thô

In [55]:
eval_deep_raw('CNN1D_raw', CNN1D_raw, X_raw, df_raw, g_raw, OUT)

    model strain             eval  Accuracy    F1   AUC    MCC
CNN1D_raw     C4        CV_random     0.495 0.365 0.515 -0.011
CNN1D_raw     C4         CV_group     0.500 0.376 0.487  0.014
CNN1D_raw     C4 independent_test     0.504 0.297 0.564  0.008
CNN1D_raw     C5        CV_random     0.631 0.344 0.695  0.183
CNN1D_raw     C5         CV_group     0.659 0.303 0.618  0.111
CNN1D_raw     C5 independent_test     0.658 0.290 0.703  0.115
CNN1D_raw    P9a        CV_random     0.686 0.686 0.742  0.377
CNN1D_raw    P9a         CV_group     0.687 0.651 0.676  0.295
CNN1D_raw    P9a independent_test     0.646 0.642 0.711  0.297

CNN1D_raw xong: 34.1 phút


,model,strain,eval,Accuracy,Accuracy_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,AUC,AUC_std,MCC,MCC_std
0,CNN1D_raw,C4,CV_random,0.495,0.014,0.285,0.138,0.535,0.298,0.365,0.192,0.515,0.035,-0.011,0.029
1,CNN1D_raw,C4,CV_group,0.500,0.071,0.327,0.124,0.537,0.278,0.376,0.165,0.487,0.027,0.014,0.039
2,CNN1D_raw,C4,independent_test,0.504,0.015,0.223,0.253,0.445,0.505,0.297,0.337,0.564,0.100,0.008,0.038
3,CNN1D_raw,C5,CV_random,0.631,0.138,0.287,0.066,0.551,0.182,0.344,0.071,0.695,0.038,0.183,0.075
4,CNN1D_raw,C5,CV_group,0.659,0.050,0.256,0.035,0.465,0.089,0.303,0.032,0.618,0.040,0.111,0.048
5,CNN1D_raw,C5,independent_test,0.658,0.174,0.253,0.091,0.414,0.255,0.290,0.094,0.703,0.049,0.115,0.099
6,CNN1D_raw,P9a,CV_random,0.686,0.029,0.680,0.024,0.702,0.047,0.686,0.032,0.742,0.015,0.377,0.060
7,CNN1D_raw,P9a,CV_group,0.687,0.052,0.646,0.055,0.697,0.077,0.651,0.051,0.676,0.034,0.295,0.058
8,CNN1D_raw,P9a,independent_test,0.646,0.029,0.641,0.036,0.653,0.095,0.642,0.040,0.711,0.024,0.297,0.055


16.3: CNN-2D với DNA thô

In [57]:
eval_deep_raw('CNN2D_raw', CNN2D_raw, X_raw, df_raw, g_raw, OUT)

    model strain             eval  Accuracy    F1   AUC   MCC
CNN2D_raw     C4        CV_random     0.540 0.505 0.589 0.096
CNN2D_raw     C4         CV_group     0.464 0.507 0.517 0.023
CNN2D_raw     C4 independent_test     0.541 0.558 0.622 0.101
CNN2D_raw     C5        CV_random     0.697 0.418 0.740 0.269
CNN2D_raw     C5         CV_group     0.657 0.316 0.649 0.123
CNN2D_raw     C5 independent_test     0.676 0.390 0.725 0.221
CNN2D_raw    P9a        CV_random     0.731 0.743 0.778 0.466
CNN2D_raw    P9a         CV_group     0.731 0.720 0.706 0.372
CNN2D_raw    P9a independent_test     0.673 0.694 0.735 0.356

CNN2D_raw xong: 31.6 phút


,model,strain,eval,Accuracy,Accuracy_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,AUC,AUC_std,MCC,MCC_std
0,CNN2D_raw,C4,CV_random,0.540,0.028,0.462,0.128,0.611,0.189,0.505,0.145,0.589,0.026,0.096,0.073
1,CNN2D_raw,C4,CV_group,0.464,0.054,0.458,0.054,0.701,0.118,0.507,0.070,0.517,0.026,0.023,0.064
2,CNN2D_raw,C4,independent_test,0.541,0.039,0.471,0.163,0.687,0.252,0.558,0.197,0.622,0.042,0.101,0.097
3,CNN2D_raw,C5,CV_random,0.697,0.033,0.333,0.043,0.597,0.067,0.418,0.051,0.740,0.042,0.269,0.053
4,CNN2D_raw,C5,CV_group,0.657,0.026,0.258,0.024,0.503,0.068,0.316,0.029,0.649,0.027,0.123,0.038
5,CNN2D_raw,C5,independent_test,0.676,0.039,0.298,0.033,0.573,0.060,0.390,0.032,0.725,0.019,0.221,0.048
6,CNN2D_raw,P9a,CV_random,0.731,0.008,0.707,0.007,0.784,0.020,0.743,0.010,0.778,0.007,0.466,0.017
7,CNN2D_raw,P9a,CV_group,0.731,0.009,0.711,0.018,0.764,0.027,0.720,0.019,0.706,0.008,0.372,0.022
8,CNN2D_raw,P9a,independent_test,0.673,0.029,0.641,0.016,0.760,0.072,0.694,0.040,0.735,0.017,0.356,0.064


16.4: mCNN với DNA thô

In [61]:
eval_deep_raw('mCNN_raw', mCNN, X_raw, df_raw, g_raw, OUT, repeats=5)

   model strain             eval  Accuracy    F1   AUC   MCC
mCNN_raw     C4        CV_random     0.577 0.589 0.628 0.179
mCNN_raw     C4         CV_group     0.505 0.527 0.520 0.053
mCNN_raw     C4 independent_test     0.534 0.645 0.643 0.103
mCNN_raw     C5        CV_random     0.716 0.477 0.753 0.344
mCNN_raw     C5         CV_group     0.648 0.380 0.698 0.200
mCNN_raw     C5 independent_test     0.672 0.428 0.724 0.275
mCNN_raw    P9a        CV_random     0.768 0.788 0.769 0.549
mCNN_raw    P9a         CV_group     0.766 0.780 0.712 0.447
mCNN_raw    P9a independent_test     0.731 0.762 0.774 0.485

mCNN_raw xong: 49.7 phút


,model,strain,eval,Accuracy,Accuracy_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,AUC,AUC_std,MCC,MCC_std
0,mCNN_raw,C4,CV_random,0.577,0.028,0.581,0.041,0.677,0.063,0.589,0.052,0.628,0.009,0.179,0.054
1,mCNN_raw,C4,CV_group,0.505,0.069,0.538,0.058,0.663,0.065,0.527,0.056,0.520,0.012,0.053,0.043
2,mCNN_raw,C4,independent_test,0.534,0.022,0.516,0.014,0.860,0.037,0.645,0.016,0.643,0.031,0.103,0.055
3,mCNN_raw,C5,CV_random,0.716,0.048,0.390,0.046,0.671,0.067,0.477,0.027,0.753,0.006,0.344,0.039
4,mCNN_raw,C5,CV_group,0.648,0.045,0.282,0.046,0.672,0.074,0.380,0.065,0.698,0.023,0.200,0.054
5,mCNN_raw,C5,independent_test,0.672,0.052,0.316,0.046,0.673,0.050,0.428,0.040,0.724,0.026,0.275,0.058
6,mCNN_raw,P9a,CV_random,0.768,0.003,0.722,0.001,0.867,0.006,0.788,0.003,0.769,0.006,0.549,0.007
7,mCNN_raw,P9a,CV_group,0.766,0.007,0.743,0.007,0.839,0.018,0.780,0.008,0.712,0.011,0.447,0.015
8,mCNN_raw,P9a,independent_test,0.731,0.015,0.676,0.018,0.873,0.015,0.762,0.009,0.774,0.013,0.485,0.024


# 17. Lưu model deep learning của DNA thô

In [63]:
# =====================================================================
# LƯU MODEL DNA THÔ (CNN1D_raw, CNN2D_raw, mCNN) cho teammate
# Chạy SAU khi train xong. Cần X_raw + df_raw trong RAM.
# =====================================================================
import os, joblib, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
OUT = r'C:\GenBBPipeline\Export'
MODEL_DIR = rf'{OUT}\models_raw'
os.makedirs(MODEL_DIR, exist_ok=True)
dev='cuda' if torch.cuda.is_available() else 'cpu'
SEED=42; BATCH=16
BINS=[s for s in ['C4','C5','P9a'] if s+'_bin' in df_raw.columns]
manifest=[]

# ---- kiến trúc (khớp raw_dna_deep + mcnn nhẹ) ----
class CNN1D_raw(nn.Module):
    def __init__(s):
        super().__init__()
        s.conv=nn.Sequential(nn.Conv1d(4,32,15,stride=4,padding=7),nn.ReLU(),nn.MaxPool1d(4),
            nn.Conv1d(32,64,9,stride=2,padding=4),nn.ReLU(),nn.MaxPool1d(4),
            nn.Conv1d(64,64,5,padding=2),nn.ReLU(),nn.AdaptiveAvgPool1d(8))
        s.head=nn.Sequential(nn.Flatten(),nn.Linear(64*8,64),nn.ReLU(),nn.Dropout(0.4),nn.Linear(64,1))
    def forward(s,x): return s.head(s.conv(x))
class CNN2D_raw(nn.Module):
    def __init__(s):
        super().__init__()
        s.conv=nn.Sequential(nn.Conv2d(1,16,(4,15),stride=(1,4),padding=(0,7)),nn.ReLU(),nn.MaxPool2d((1,4)),
            nn.Conv2d(16,32,(1,9),stride=(1,2),padding=(0,4)),nn.ReLU(),nn.AdaptiveAvgPool2d((1,8)))
        s.head=nn.Sequential(nn.Flatten(),nn.Linear(32*8,64),nn.ReLU(),nn.Dropout(0.4),nn.Linear(64,1))
    def forward(s,x): return s.head(s.conv(x.unsqueeze(1)))
class mCNN(nn.Module):
    def __init__(s,window_sizes=(8,24,48),num_feature=4,num_filters=128,num_hidden=512):
        super().__init__()
        s.convs=nn.ModuleList([nn.Conv2d(1,num_filters,(w,num_feature)) for w in window_sizes])
        s.dropout=nn.Dropout(0.7); s.fc1=nn.Linear(num_filters*len(window_sizes),num_hidden); s.fc2=nn.Linear(num_hidden,1)
    def forward(s,x):
        if x.dim()==3: x=x.transpose(1,2).unsqueeze(1)
        feats=[F.relu(c(x)).max(dim=2).values.squeeze(-1) for c in s.convs]
        return s.fc2(F.relu(s.fc1(s.dropout(torch.cat(feats,1)))))

RAW={'CNN1D_raw':CNN1D_raw,'CNN2D_raw':CNN2D_raw,'mCNN_raw':mCNN}
print('=== Lưu model DNA thô (train full + save) ===')
for s in BINS:
    y=df_raw[s+'_bin'].values.astype(int)
    npos=max(y.sum(),1); pw=torch.tensor([(len(y)-npos)/npos],dtype=torch.float32,device=dev)
    Xt=torch.tensor(X_raw,dtype=torch.float32); yt=torch.tensor(y,dtype=torch.float32)
    for name,cls in RAW.items():
        torch.manual_seed(SEED); m=cls().to(dev); m.train()
        opt=torch.optim.Adam(m.parameters(),lr=1e-3,weight_decay=1e-3); lf=nn.BCEWithLogitsLoss(pos_weight=pw)
        for _ in range(60):
            perm=torch.randperm(len(Xt))
            for i in range(0,len(Xt),BATCH):
                idx=perm[i:i+BATCH]; xb=Xt[idx].to(dev); yb=yt[idx].to(dev)
                opt.zero_grad(); lf(m(xb).squeeze(-1),yb).backward(); opt.step()
        torch.save({'state_dict':m.state_dict(),'arch':name,'input_shape':list(X_raw.shape[1:]),
                    'maxlen_gene':X_raw.shape[2]//5,'strain':s}, rf'{MODEL_DIR}\{name}_{s}.pt')
        manifest.append(dict(file=f'{name}_{s}.pt',model=name,strain=s,framework='torch'))
    print(f'  {s}: {list(RAW.keys())}')

pd.DataFrame(manifest).to_csv(rf'{MODEL_DIR}\manifest.csv',index=False)
readme=f"""# MODELS DNA THÔ (CNN1D/2D, mCNN) — one-hot

## Input: X_raw (N, 4, {X_raw.shape[2]}) = one-hot 5 gene, mỗi gene {X_raw.shape[2]//5}bp
   Gene: xa5, xa13_SWEET11, xa25_SWEET13, Xa21, Xa1 (nối); kênh A/C/G/T
   KHÔNG chuẩn hóa (one-hot đã 0/1). Gene thiếu -> pad 0.
## mCNN: kiến trúc gốc paper thầy (multi-window Conv2D + 1-max pool), 3 cửa sổ (8,24,48)
   mCNN lặp 5 lần (thay 20) do chi phí tính toán cao.
## File: {{model}}_{{strain}}.pt (state_dict + arch + input_shape)
## LƯU Ý: deep learning trên chuỗi thô < classical. Xem results_ALL_3branch.csv.
"""
open(rf'{MODEL_DIR}\README_raw.txt','w',encoding='utf-8').write(readme)
import shutil; shutil.make_archive(rf'{OUT}\models_raw','zip',MODEL_DIR)
print(f'\n=== XONG {len(manifest)} model DNA thô ===')
print(f'>>> {OUT}\\models_raw.zip  — NHỚ TẢI VỀ trước khi trả máy! <<<')

=== Lưu model DNA thô (train full + save) ===
  C4: ['CNN1D_raw', 'CNN2D_raw', 'mCNN_raw']
  C5: ['CNN1D_raw', 'CNN2D_raw', 'mCNN_raw']
  P9a: ['CNN1D_raw', 'CNN2D_raw', 'mCNN_raw']

=== XONG 9 model DNA thô ===
>>> C:\GenBBPipeline\Export\models_raw.zip  — NHỚ TẢI VỀ trước khi trả máy! <<<


17.2: Xuất file tổng hợp

In [62]:
# =====================================================================
# TỔNG HỢP 3 NHÁNH: SNP vs EMBEDDING vs DNA THÔ
# Bảng so sánh cuối cho luận văn + ISDS
# =====================================================================
import pandas as pd, glob, os, numpy as np
OUT = r'C:\GenBBPipeline\Export'
files = [f for f in glob.glob(rf'{OUT}\train_*.csv') if 'SMOKE' not in f]

DEEP={'MLP','CNN1D','CNN2D','mCNN'}
def branch_of(name):
    if name.endswith('_SNP'): return 'SNP'
    if name.endswith('_raw'): return 'DNA_tho'
    return 'embedding'
def base_of(name):
    return name.replace('_SNP','').replace('_emb','').replace('_raw','')

parts=[]
for f in files:
    name=os.path.basename(f).replace('train_','').replace('.csv','')
    d=pd.read_csv(f); d=d[[c for c in d.columns if not c.endswith('_std')]].copy()
    d['branch']=branch_of(name); base=base_of(name)
    d['family']='deep' if base in DEEP else 'classical'
    d['model_base']=base
    parts.append(d)
RES=pd.concat(parts,ignore_index=True)
RES.to_csv(rf'{OUT}\results_ALL_3branch.csv',index=False)
print(f'Gộp {len(files)} file | {len(RES)} dòng | nhánh: {sorted(RES.branch.unique())}\n')

# BẢNG 1: CV_group MCC — nhánh × loại
print('='*70); print('BẢNG 1 — CV_group MCC TB: nhánh × loại model'); print('='*70)
g=RES[RES['eval']=='CV_group']
print(g.pivot_table(index='branch',columns='family',values='MCC',aggfunc='mean').round(3).to_string())

# BẢNG 2: independent_test AUC — nhánh × loại
print('\n'+'='*70); print('BẢNG 2 — Independent test AUC TB: nhánh × loại'); print('='*70)
t=RES[RES['eval']=='independent_test']
print(t.pivot_table(index='branch',columns='family',values='AUC',aggfunc='mean').round(3).to_string())

# BẢNG 3: xếp hạng model (CV_group MCC, TB 3 chủng)
print('\n'+'='*70); print('BẢNG 3 — Xếp hạng model (CV_group MCC, TB 3 chủng)'); print('='*70)
print(g.groupby(['branch','model_base']).MCC.mean().round(3).sort_values(ascending=False).to_string())

# BẢNG 4: rò rỉ theo nhánh
print('\n'+'='*70); print('BẢNG 4 — Rò rỉ (random-group AUC) theo nhánh'); print('='*70)
leak=RES[RES['eval'].isin(['CV_random','CV_group'])].groupby(['branch','eval']).AUC.mean().unstack().round(3)
leak['gap']=(leak['CV_random']-leak['CV_group']).round(3)
print(leak.to_string())

# BẢNG 5: model tốt nhất mỗi nhánh (test AUC)
print('\n'+'='*70); print('BẢNG 5 — Model tốt nhất mỗi nhánh (test AUC, TB 3 chủng)'); print('='*70)
best=t.groupby(['branch','model_base'])[['AUC','MCC','F1']].mean().round(3)
for br in sorted(RES.branch.unique()):
    print(f'\n--- {br} ---')
    print(best.loc[br].sort_values('AUC',ascending=False).head(3).to_string())

# BẢNG 6: classical vs deep tổng thể
print('\n'+'='*70); print('BẢNG 6 — Classical vs Deep (TB mọi nhánh)'); print('='*70)
for ev in ['CV_group','independent_test']:
    sub=RES[RES['eval']==ev].groupby('family')[['AUC','MCC']].mean().round(3)
    print(f'\n[{ev}]'); print(sub.to_string())

print(f'\nĐã lưu results_ALL_3branch.csv')

Gộp 19 file | 171 dòng | nhánh: ['DNA_tho', 'SNP', 'embedding']

BẢNG 1 — CV_group MCC TB: nhánh × loại model
family     classical   deep
branch                     
DNA_tho          NaN  0.182
SNP            0.200  0.144
embedding      0.183  0.149

BẢNG 2 — Independent test AUC TB: nhánh × loại
family     classical   deep
branch                     
DNA_tho          NaN  0.689
SNP            0.828  0.744
embedding      0.758  0.686

BẢNG 3 — Xếp hạng model (CV_group MCC, TB 3 chủng)
branch     model_base  
SNP        RandomForest    0.261
DNA_tho    mCNN            0.233
embedding  RandomForest    0.218
           XGBoost         0.204
SNP        XGBoost         0.204
embedding  MLP             0.185
           CNN2D           0.185
SNP        LogReg          0.173
DNA_tho    CNN2D           0.173
embedding  CNN1D           0.169
SNP        SVM             0.160
embedding  LogReg          0.158
           SVM             0.153
SNP        MLP             0.152
           mCNN         

# Lưu Môi trường và các yêu cầu thư viện

Lưu thư viện

In [64]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True,
    text=True
)

requirements = result.stdout

with open(
    "C:/GenBBPipeline/requirements.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(requirements)

print(requirements)

annotated-doc==0.0.5
anyio==4.14.2
asttokens @ file:///home/conda/feedstock_root/build_artifacts/asttokens_1783975656336/work
bed-reader==1.1.0
biopython==1.88
certifi==2026.7.22
click==8.4.2
colorama @ file:///home/conda/feedstock_root/build_artifacts/colorama_1733218098505/work
comm @ file:///home/conda/feedstock_root/build_artifacts/bld/rattler-build_comm_1753453984/work
debugpy @ file:///D:/bld/bld/rattler-build_debugpy_1780390184/work
decorator @ file:///home/conda/feedstock_root/build_artifacts/decorator_1779115150916/work
einops==0.8.2
exceptiongroup @ file:///home/conda/feedstock_root/build_artifacts/exceptiongroup_1763918002538/work
executing @ file:///home/conda/feedstock_root/build_artifacts/executing_1756729339227/work
filelock==3.29.0
fsspec==2026.4.0
h11==0.16.0
hf-xet==1.6.0
httpcore==1.0.9
httpx==0.28.1
huggingface_hub==1.27.0
idna==3.18
ipykernel @ file:///D:/bld/bld/rattler-build_ipykernel_1781101760/work
ipython @ file:///D:/bld/bld/rattler-build_ipython_1748711254/w

Lưu các môi trường

In [67]:
import platform
import torch
import sys

info = f"""
Python:
{sys.version}

Platform:
{platform.platform()}

PyTorch:
{torch.__version__}

CUDA available:
{torch.cuda.is_available()}

CUDA version:
{torch.version.cuda}

GPU:
{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}
"""

with open(
    "C:/GenBBPipeline/environment.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(info)

print(info)


Python:
3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]

Platform:
Windows-10-10.0.19045-SP0

PyTorch:
2.11.0+cu128

CUDA available:
True

CUDA version:
12.8

GPU:
NVIDIA GeForce RTX 5060 Ti

